# Notebook 09 — MURA Agentic AI

## Agentic Decision Support for Musculoskeletal Radiographs

### Project Objective

Develop an Agentic AI system that integrates a previously trained ResNet18 classifier with technical quality-control tools, uncertainty assessment, explainability methods, and structured reporting.

The system processes de-identified MURA wrist radiographic studies and generates evidence-based workflow recommendations.

### Scientific Motivation

The fine-tuned ResNet18 achieved the following performance on the held-out official MURA validation dataset:

| Metric | Result |
|---|---:|
| ROC-AUC | 0.9060 |
| Sensitivity | 90.72% |
| Specificity | 62.14% |
| False-negative studies | 9 |
| False-positive studies | 53 |

Further analysis showed that five of the nine false-negative studies contained at least one radiograph with an abnormality probability above the predefined classification threshold, although their mean study-level probability remained below it.

These findings motivate additional investigation of image-level disagreement, prediction uncertainty, and aggregation-related limitations.

### Why Agentic AI?

A conventional classifier generates predictions but does not independently investigate contradictory evidence or select additional analytical tools.

The proposed agent will:

1. Examine classifier predictions and image-level consistency.
2. Identify ambiguous or potentially unreliable results.
3. Select additional tools when warranted.
4. Integrate evidence from multiple analytical methods.
5. Generate structured, auditable reports.
6. Recommend appropriate human-review workflows.

### Agent Architecture

**Input:** De-identified wrist radiographic study containing one or more images.

**Mandatory processing:**
- Input validation and technical QC.
- Frozen ResNet18 classification.
- Image-level and study-level probability calculation.
- Cross-view consistency assessment.

**Agent-controlled tools:**
- Test-time augmentation for uncertainty assessment.
- Grad-CAM for visual explanation.
- De-identified study metadata retrieval.

**Output:**
- Study-level abnormality probability and classification.
- Image-level predictions and consistency findings.
- Technical QC results.
- Additional tool findings, when requested.
- Structured report with evidence and limitations.
- Workflow recommendation.

### Safety Constraints

- The classifier and predefined threshold remain unchanged.
- The agent cannot override deterministic guardrails.
- No specific diagnosis or confirmed fracture is generated.
- Unsupported, invalid, or insufficiently reliable inputs require appropriate escalation.
- Grad-CAM is not considered validated lesion localization.
- Final clinical interpretation remains with the radiologist or responsible clinician.
- No identifiable patient information or live EHR access is included.

### Agent Evaluation

The agent will be compared with a deterministic rule-based system using the same classifier, threshold, guardrails, and predefined evaluation cases.

Evaluation criteria include:

- Tool-selection accuracy.
- Recognition of conflicting evidence.
- Appropriate uncertainty assessment.
- Report groundedness.
- Correct human escalation.
- Safety violations.
- Unnecessary tool calls and execution time.

### Expected Outcome

A functional, auditable Agentic AI prototype demonstrating whether adaptive tool orchestration provides measurable benefits beyond a conventional classifier and rule-based workflow.

**Development environment:** Google Colab  
**Dataset:** MURA-v1.1 XR_WRIST  
**Classifier:** Fine-tuned ResNet18  
**Agent framework:** To be selected during implementation

## Part A — Agent Environment Initialization

### Objective

Initialize the Agentic AI development environment and verify access to the frozen ResNet18 model and official validation outputs.

### Project Organization

The agent will use a dedicated directory:

`DATA/MURA_AGENTIC_AI/`

This directory will contain:

- Agent configuration files.
- Tool definitions and outputs.
- Agent execution logs.
- Structured reports.
- Evaluation scenarios.
- Agent-versus-rule-based comparison results.

### Model Integrity

The agent will use the previously selected ResNet18 checkpoint and classification threshold.

No additional model training or threshold optimization will be performed.

### Expected Outcome

A verified project environment with access to the frozen classifier and a separate directory for agent development.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import torch

# ---------------------------------------
# 1. Mount Google Drive
# ---------------------------------------

drive.mount("/content/drive")

# ---------------------------------------
# 2. Define project directories
# ---------------------------------------

DATA_DIR = Path("/content/drive/MyDrive/DATA")

MODEL_DIR = DATA_DIR / "RESNET18_FINE_TUNING"

VALIDATION_DIR = (
    DATA_DIR / "MURA_OFFICIAL_VALIDATION"
)

AGENT_DIR = DATA_DIR / "MURA_AGENTIC_AI"

AGENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------
# 3. Create agent subdirectories
# ---------------------------------------

AGENT_SUBDIRS = {
    "config": AGENT_DIR / "config",
    "tools": AGENT_DIR / "tools",
    "logs": AGENT_DIR / "logs",
    "reports": AGENT_DIR / "reports",
    "evaluation": AGENT_DIR / "evaluation",
    "figures": AGENT_DIR / "figures"
}

for directory in AGENT_SUBDIRS.values():
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ---------------------------------------
# 4. Verify frozen model files
# ---------------------------------------

CHECKPOINT_PATH = (
    MODEL_DIR / "resnet18_best.pt"
)

THRESHOLD_PATH = (
    MODEL_DIR /
    "resnet18_internal_val_threshold_metrics.json"
)

assert CHECKPOINT_PATH.exists(), (
    "Frozen ResNet18 checkpoint not found."
)

assert THRESHOLD_PATH.exists(), (
    "Threshold configuration not found."
)

# ---------------------------------------
# 5. Load fixed threshold
# ---------------------------------------

with open(THRESHOLD_PATH, "r") as f:
    threshold_metrics = json.load(f)

SELECTED_THRESHOLD = (
    threshold_metrics["selected_threshold"]
)

# ---------------------------------------
# 6. Verify official validation files
# ---------------------------------------

OFFICIAL_METRICS_PATH = (
    VALIDATION_DIR /
    "resnet18_official_performance_metrics.json"
)

assert OFFICIAL_METRICS_PATH.exists(), (
    "Official validation results not found."
)

with open(OFFICIAL_METRICS_PATH, "r") as f:
    official_metrics = json.load(f)

# ---------------------------------------
# 7. Verify runtime
# ---------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# ---------------------------------------
# 8. Display summary
# ---------------------------------------

print("NOTEBOOK 09 — AGENTIC AI INITIALIZATION")
print("-" * 60)

print("\nENVIRONMENT")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nMODEL")
print("Checkpoint:", CHECKPOINT_PATH.name)
print("Fixed threshold:", round(SELECTED_THRESHOLD, 6))

print("\nOFFICIAL VALIDATION")
print(
    "ROC-AUC:",
    round(official_metrics["roc_auc"], 4)
)
print(
    "Sensitivity:",
    round(official_metrics["sensitivity"], 4)
)
print(
    "Specificity:",
    round(official_metrics["specificity"], 4)
)

print("\nAGENT DIRECTORY")
print(AGENT_DIR)

print("\nSUBDIRECTORIES")

for name, directory in AGENT_SUBDIRS.items():
    print(f"{name}: {directory.name}")

print("\nInitialization: PASSED")

## Part A — Step 3: Agent Specification and Tool Contracts

### Agent Identity

**Name:** MURA Wrist Decision-Support Agent

**Role:** AI research assistant for musculoskeletal radiograph analysis and workflow decision support.

### Agent Objective

The agent integrates outputs from a frozen ResNet18 classifier with technical quality-control, cross-view consistency, uncertainty assessment, and explainability tools.

It dynamically selects additional analytical tools when necessary and generates structured, auditable reports.

### Agent Inputs

- De-identified study identifier.
- One or more wrist radiographs.
- Technical image metadata.
- Frozen ResNet18 predictions.
- Fixed classification threshold.

### Agent-Accessible Tools

| Tool | Description |
|---|---|
| `technical_qc` | Validates image files and evaluates basic technical characteristics |
| `classify_study` | Calculates image-level and mean study-level abnormality probabilities |
| `view_consistency` | Identifies disagreement between individual image probabilities |
| `tta_uncertainty` | Evaluates prediction stability using predefined test-time transformations |
| `gradcam_explainer` | Generates visual explanations of model predictions |
| `study_metadata` | Retrieves de-identified study information |
| `report_generator` | Produces structured reports from verified evidence |

### Agent Decision Process

1. Receive a de-identified wrist study.
2. Execute mandatory technical QC.
3. Run the frozen classifier if the input passes mandatory validation.
4. Evaluate image-level agreement.
5. Identify ambiguous or inconsistent evidence.
6. Select additional tools when appropriate.
7. Integrate tool outputs.
8. Produce a structured report and permitted workflow recommendation.

### Allowed Workflow Recommendations

- **Routine workflow:** No additional concern identified by the research workflow.
- **Priority human review:** Potentially abnormal findings or significant view disagreement.
- **Human escalation:** Insufficient, contradictory, or unreliable evidence.
- **Unsupported input:** Input does not satisfy mandatory requirements.

These recommendations are research-prototype outputs, not validated clinical triage instructions.

### Safety Constraints

- The frozen ResNet18 model and threshold cannot be modified.
- The agent cannot independently diagnose fractures or other specific conditions.
- Technical QC cannot be represented as clinical image-quality assessment.
- Grad-CAM cannot be represented as validated lesion localization.
- Unsupported or invalid input must not receive a forced classification.
- The agent must not invent model outputs or clinical findings.
- Final clinical interpretation remains with a qualified clinician.

### Audit Requirements

Every agent execution must record:

- Study identifier.
- Model and threshold version.
- Tools executed.
- Tool outputs.
- Agent reasoning summary.
- Workflow recommendation.
- Explanation supporting the recommendation.
- Execution timestamp.

### Evaluation Strategy

The agent will be evaluated against a deterministic rule-based workflow using identical input studies and the same frozen classifier.

The comparison will assess tool selection, handling of ambiguous evidence, grounded reporting, escalation behavior, and safety compliance.

**Save the agent specification**
This cell creates a machine-readable configuration file that we will use when implementing the agent.

In [ ]:
import json
from datetime import datetime, timezone

print("NOTEBOOK 09 — AGENT SPECIFICATION")
print("-" * 60)

agent_spec = {
    "agent_name": "MURA Wrist Decision-Support Agent",
    "version": "1.0",
    "purpose": (
        "Research-oriented orchestration of wrist radiograph "
        "classification, uncertainty analysis, explainability, "
        "and evidence-grounded reporting."
    ),
    "model": {
        "architecture": "ResNet18",
        "checkpoint": str(CHECKPOINT_PATH),
        "selected_epoch": 7,
        "threshold": float(SELECTED_THRESHOLD),
        "aggregation": "mean_image_probability",
        "frozen": True,
        "task": "normal_vs_abnormal"
    },
    "tools": {
        "technical_qc": {
            "execution": "mandatory",
            "description": "Technical image validation"
        },
        "classify_study": {
            "execution": "mandatory_after_qc",
            "description": "Frozen ResNet18 inference"
        },
        "view_consistency": {
            "execution": "mandatory_after_classification",
            "description": "Cross-view probability comparison"
        },
        "tta_uncertainty": {
            "execution": "agent_selected",
            "description": "Prediction stability assessment"
        },
        "gradcam_explainer": {
            "execution": "agent_selected",
            "description": "Model activation visualization"
        },
        "study_metadata": {
            "execution": "agent_selected",
            "description": "De-identified study information"
        },
        "report_generator": {
            "execution": "mandatory",
            "description": "Structured evidence-grounded report"
        }
    },
    "allowed_actions": [
        "routine_workflow",
        "priority_human_review",
        "human_escalation",
        "unsupported_input"
    ],
    "safety_constraints": {
        "model_weights_immutable": True,
        "threshold_immutable": True,
        "no_specific_diagnosis": True,
        "no_unverified_clinical_claims": True,
        "no_patient_identifiers": True,
        "human_final_decision": True
    },
    "evaluation": {
        "baseline": "deterministic_rule_based_policy",
        "metrics": [
            "tool_selection_accuracy",
            "view_disagreement_detection",
            "uncertainty_handling",
            "report_groundedness",
            "escalation_accuracy",
            "safety_violations",
            "unnecessary_tool_calls",
            "execution_time"
        ]
    },
    "created_at": datetime.now(
        timezone.utc
    ).isoformat()
}

# Save configuration
config_path = (
    AGENT_SUBDIRS["config"] /
    "mura_agent_specification.json"
)

with open(config_path, "w") as f:
    json.dump(agent_spec, f, indent=4)

# Verify saved configuration
with open(config_path, "r") as f:
    loaded_spec = json.load(f)

assert loaded_spec["model"]["frozen"] is True
assert loaded_spec["model"]["threshold"] == SELECTED_THRESHOLD
assert len(loaded_spec["tools"]) == 7
assert len(loaded_spec["allowed_actions"]) == 4

print("\nAGENT")
print("Name:", loaded_spec["agent_name"])
print("Version:", loaded_spec["version"])

print("\nMODEL")
print("Architecture:", loaded_spec["model"]["architecture"])
print("Frozen:", loaded_spec["model"]["frozen"])
print("Threshold:", loaded_spec["model"]["threshold"])

print("\nTOOLS")
for name, details in loaded_spec["tools"].items():
    print(f"{name}: {details['execution']}")

print("\nALLOWED ACTIONS")
for action in loaded_spec["allowed_actions"]:
    print("-", action)

print("\nCONFIGURATION SAVED")
print(config_path)

print("\nStep 3: PASSED")

## Step 4 — Technical Image Quality Control

### Objective

Implement a deterministic technical quality-control tool for de-identified MURA wrist radiographs.

The tool evaluates image integrity and basic technical characteristics before model inference.

### Quality-Control Checks

- Image readability and decoding.
- Image dimensions.
- Image brightness.
- Image contrast.
- Detection of blank or near-uniform images.
- Presence of at least one valid image in the study.

### QC Outcomes

**PASS:** No technical issues detected.

**WARNING:** Potential technical limitations identified; the image remains processable.

**FAIL:** A blocking technical problem prevents reliable processing.

### Safety Requirements

- Invalid or unreadable images must not be silently ignored.
- The agent cannot override blocking QC failures.
- Technical QC does not establish clinical diagnostic adequacy.
- QC findings must be included in the agent's audit trail.
- No patient-identifying information is required.

### Expected Output

A structured QC result containing:

- Image identifier.
- Image dimensions.
- Mean brightness.
- Contrast estimate.
- QC status.
- Identified issues.

### Validation Strategy

The tool will be tested using valid MURA radiographs and deliberately generated invalid or degraded test images.

The tests will verify that blocking failures and warnings are identified consistently.

**Implement the QC tool**

In [ ]:
import io
import numpy as np
from PIL import Image, UnidentifiedImageError

print("NOTEBOOK 09 — STEP 4: TECHNICAL QC TOOL")
print("-" * 60)


def technical_qc(image_bytes, image_id="UNKNOWN"):
    """
    Deterministic technical QC for a radiographic image.

    Input:
        image_bytes: Encoded image file as bytes.
        image_id: De-identified image identifier.

    Output:
        Structured QC dictionary.

    Note:
        Thresholds are engineering heuristics,
        not clinically validated QC criteria.
    """

    result = {
        "image_id": image_id,
        "status": "FAIL",
        "width": None,
        "height": None,
        "mean_brightness": None,
        "contrast_std": None,
        "issues": []
    }

    # ---------------------------------------
    # 1. Validate input
    # ---------------------------------------

    if not isinstance(image_bytes, bytes) or len(image_bytes) == 0:
        result["issues"].append("empty_or_invalid_input")
        return result

    # ---------------------------------------
    # 2. Decode image
    # ---------------------------------------

    try:
        with Image.open(io.BytesIO(image_bytes)) as img:
            image = img.convert("L")
            pixels = np.asarray(
                image,
                dtype=np.float32
            )

    except (UnidentifiedImageError, OSError, ValueError):
        result["issues"].append("unreadable_image")
        return result

    # ---------------------------------------
    # 3. Check dimensions
    # ---------------------------------------

    height, width = pixels.shape

    result["width"] = int(width)
    result["height"] = int(height)

    if width < 128 or height < 128:
        result["issues"].append("image_too_small")

    # ---------------------------------------
    # 4. Brightness and contrast
    # ---------------------------------------

    mean_brightness = float(pixels.mean())
    contrast_std = float(pixels.std())

    result["mean_brightness"] = mean_brightness
    result["contrast_std"] = contrast_std

    if mean_brightness < 5:
        result["issues"].append("extremely_dark")

    if mean_brightness > 250:
        result["issues"].append("extremely_bright")

    if contrast_std < 5:
        result["issues"].append("near_uniform_image")

    # ---------------------------------------
    # 5. Assign status
    # ---------------------------------------

    blocking_issues = {
        "image_too_small",
        "near_uniform_image"
    }

    if any(
        issue in blocking_issues
        for issue in result["issues"]
    ):
        result["status"] = "FAIL"

    elif result["issues"]:
        result["status"] = "WARNING"

    else:
        result["status"] = "PASS"

    return result


# ---------------------------------------
# 6. Create synthetic test images
# ---------------------------------------

def encode_png(array):
    image = Image.fromarray(
        array.astype(np.uint8)
    )

    buffer = io.BytesIO()
    image.save(buffer, format="PNG")

    return buffer.getvalue()


rng = np.random.default_rng(42)

test_cases = {
    "valid_image": encode_png(
        rng.integers(
            0, 256,
            size=(512, 512),
            dtype=np.uint8
        )
    ),
    "blank_image": encode_png(
        np.zeros((512, 512), dtype=np.uint8)
    ),
    "small_image": encode_png(
        rng.integers(
            0, 256,
            size=(64, 64),
            dtype=np.uint8
        )
    ),
    "corrupted_image": b"not_a_valid_image",
    "empty_image": b""
}

# ---------------------------------------
# 7. Run QC tests
# ---------------------------------------

qc_results = {}

for name, image_bytes in test_cases.items():

    result = technical_qc(
        image_bytes,
        image_id=name
    )

    qc_results[name] = result

    print(
        f"{name:20s} "
        f"Status: {result['status']:8s} "
        f"Issues: {result['issues']}"
    )

# ---------------------------------------
# 8. Verify expected behavior
# ---------------------------------------

assert qc_results["valid_image"]["status"] == "PASS"
assert qc_results["blank_image"]["status"] == "FAIL"
assert qc_results["small_image"]["status"] == "FAIL"
assert qc_results["corrupted_image"]["status"] == "FAIL"
assert qc_results["empty_image"]["status"] == "FAIL"

print("\nStep 4: PASSED")

## Step 5 — Technical QC Evaluation on Real MURA Images

### Objective

Evaluate the technical QC tool using real MURA wrist radiographs.

### Dataset

Official MURA-v1.1 XR_WRIST validation images.

- 659 images
- 237 studies
- 207 patients

### Methodology

1. Load the existing official validation image lookup table.
2. Retrieve each image from the original MURA archive.
3. Apply the deterministic technical QC tool.
4. Summarize PASS, WARNING, and FAIL outcomes.
5. Examine image dimensions, brightness, and contrast.
6. Save image-level QC results for subsequent agent development.

### Important Considerations

QC thresholds are preliminary engineering heuristics.

A QC failure does not necessarily indicate that an image is clinically inadequate.

The held-out dataset will not be used to modify the frozen ResNet18 classifier or its classification threshold.

Any QC threshold revisions informed by this dataset will be considered exploratory and require separate validation.

### Expected Outcome

A structured dataset of technical QC findings and a summary of the tool's behavior on real radiographs.

In [ ]:
import pandas as pd
import zipfile

print("NOTEBOOK 09 — STEP 5A: MURA IMAGE ACCESS")
print("-" * 60)

# ---------------------------------------
# 1. Locate official validation lookup
# ---------------------------------------

LOOKUP_PATH = (
    VALIDATION_DIR /
    "official_wrist_image_lookup.csv"
)

assert LOOKUP_PATH.exists(), (
    "Official image lookup file not found."
)

lookup_df = pd.read_csv(LOOKUP_PATH)

print("\nIMAGE LOOKUP")
print("Rows:", len(lookup_df))
print("Columns:", lookup_df.columns.tolist())

print("\nFIRST THREE RECORDS")
print(lookup_df.head(3).to_string(index=False))

# ---------------------------------------
# 2. Locate MURA archive
# ---------------------------------------

MURA_ZIP_PATH = (
    DATA_DIR /
    "MURA-v1.1_files.zip"
)

assert MURA_ZIP_PATH.exists(), (
    "MURA ZIP archive not found."
)

print("\nMURA ARCHIVE")
print("Archive found:", MURA_ZIP_PATH.name)

# ---------------------------------------
# 3. Inspect ZIP structure
# ---------------------------------------

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    archive_names = zf.namelist()

    print("\nARCHIVE")
    print("Total entries:", len(archive_names))

    print("\nFIRST FIVE ENTRIES")
    for name in archive_names[:5]:
        print(name)

# ---------------------------------------
# 4. Inspect lookup image paths
# ---------------------------------------

print("\nSAMPLE IMAGE PATHS")

for path in lookup_df["image_path"].head(3):
    print(path)

print("\nStep 5A: PASSED")

## Step 5B — Technical QC Evaluation on 659 MURA Wrist Images

### Objective

Apply the technical quality-control tool to all 659 wrist radiographs from the official MURA validation dataset.

### Procedure

1. Read each radiograph directly from the MURA ZIP archive.
2. Apply the deterministic `technical_qc` function.
3. Record image dimensions, brightness, contrast, QC status, and detected issues.
4. Summarize the distribution of QC outcomes.
5. Save the results for subsequent agent development.

### Quality-Control Policy

**PASS:** No technical issues detected.

**WARNING:** A potential technical limitation was identified, but the image remains processable.

**FAIL:** A blocking technical problem was detected.

### Interpretation

The QC thresholds are preliminary engineering heuristics, not clinically validated image-quality criteria.

The official validation dataset is used here for exploratory technical compatibility testing. Any subsequent modifications to QC thresholds will require evaluation on separate data.

### Expected Outputs

- `mura_official_technical_qc.csv`
- Summary of QC status distribution.
- Summary of image dimensions, brightness, and contrast.
- Frequency of individual QC issues.

These results will help determine whether the QC tool is suitable for integration into the agent.

In [ ]:
from collections import Counter
import pandas as pd
import zipfile

print("NOTEBOOK 09 — STEP 5B: REAL MURA QC")
print("-" * 60)

# ---------------------------------------
# 1. Initialize results
# ---------------------------------------

qc_records = []

# ---------------------------------------
# 2. Open archive once
# ---------------------------------------

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    archive_names = set(zf.namelist())

    for row in lookup_df.itertuples(index=False):

        image_id = row.image_id
        image_path = row.image_path

        # Verify image exists in archive
        if image_path not in archive_names:

            result = {
                "image_id": image_id,
                "status": "FAIL",
                "width": None,
                "height": None,
                "mean_brightness": None,
                "contrast_std": None,
                "issues": ["image_missing_from_archive"]
            }

        else:

            try:
                image_bytes = zf.read(image_path)

                result = technical_qc(
                    image_bytes,
                    image_id=image_id
                )

            except Exception as e:

                result = {
                    "image_id": image_id,
                    "status": "FAIL",
                    "width": None,
                    "height": None,
                    "mean_brightness": None,
                    "contrast_std": None,
                    "issues": [
                        "archive_read_error"
                    ]
                }

        result["image_path"] = image_path

        qc_records.append(result)

# ---------------------------------------
# 3. Create DataFrame
# ---------------------------------------

qc_df = pd.DataFrame(qc_records)

assert len(qc_df) == len(lookup_df)

# Convert issues to readable text
qc_df["issues"] = qc_df["issues"].apply(
    lambda issues: "; ".join(issues)
)

# ---------------------------------------
# 4. Save results
# ---------------------------------------

QC_OUTPUT_PATH = (
    AGENT_DIR /
    "evaluation" /
    "mura_official_technical_qc.csv"
)

qc_df.to_csv(
    QC_OUTPUT_PATH,
    index=False
)

# ---------------------------------------
# 5. Summarize QC status
# ---------------------------------------

print("\nQC STATUS DISTRIBUTION")

status_counts = (
    qc_df["status"]
    .value_counts()
    .reindex(
        ["PASS", "WARNING", "FAIL"],
        fill_value=0
    )
)

print(status_counts.to_string())

# ---------------------------------------
# 6. Summarize image characteristics
# ---------------------------------------

print("\nIMAGE CHARACTERISTICS")

columns = [
    "width",
    "height",
    "mean_brightness",
    "contrast_std"
]

print(
    qc_df[columns]
    .describe()
    .round(2)
    .to_string()
)

# ---------------------------------------
# 7. Count QC issues
# ---------------------------------------

issue_counter = Counter()

for issues in qc_df["issues"]:

    if issues:
        for issue in issues.split("; "):
            issue_counter[issue] += 1

print("\nQC ISSUE FREQUENCIES")

if issue_counter:

    for issue, count in issue_counter.most_common():
        print(f"{issue}: {count}")

else:
    print("No QC issues detected.")

# ---------------------------------------
# 8. Verify results
# ---------------------------------------

assert set(qc_df["status"]).issubset(
    {"PASS", "WARNING", "FAIL"}
)

assert len(qc_df) == 659

print("\nOUTPUT SAVED")
print(QC_OUTPUT_PATH)

print("\nStep 5B: PASSED")

The absence of QC warnings does not establish that all radiographs are clinically adequate.

In particular, the current technical QC tool does not evaluate:

- Anatomical region (wrist versus other anatomical regions).
- Anatomical positioning.
- Motion artifacts.
- Diagnostic exposure adequacy.
- Completeness of anatomical coverage.
- Visibility of specific abnormalities.

**Anatomical region identification will be addressed separately in Step 7 through a dedicated anatomical scope validator.** This validator will act as an additional mandatory safeguard before ResNet18 classification.

Neither technical QC nor anatomical scope validation establishes the clinical diagnostic adequacy of a radiograph.

**Tool 1: technical_qc	Checks image readability, dimensions, brightness, and contrast	✅ Implemented and tested on 659 images**

## Step 6 — Frozen ResNet18 Classification Tool

### Objective

Integrate the previously trained ResNet18 classifier into the Agentic AI system without modifying the model or its decision threshold.

### Model Configuration

- Architecture: ResNet18
- Task: Binary classification (normal versus abnormal)
- Input: MURA wrist radiographs
- Input size: 320 × 320 pixels
- Normalization: ImageNet mean and standard deviation
- Study aggregation: Arithmetic mean of image-level probabilities
- Classification threshold: 0.0600893441587686
- Model parameters: Frozen

### Tool Responsibilities

The classification tool will:

1. Receive validated radiographic images.
2. Apply the original image preprocessing.
3. Generate image-level abnormality probabilities.
4. Calculate the mean study-level probability.
5. Apply the frozen classification threshold.
6. Return structured results to the agent.

### Safety Requirements

- No model retraining or fine-tuning.
- No threshold recalibration.
- No modification of the study aggregation method.
- No classification of studies that fail mandatory technical validation.
- The agent cannot override model predictions.

### Expected Outcome

A reusable classification tool that reproduces the original ResNet18 inference workflow and can be called by the Agentic AI system.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms

print("NOTEBOOK 09 — STEP 6A: LOAD FROZEN RESNET18")
print("-" * 60)

# ---------------------------------------
# 1. Reconstruct model architecture
# ---------------------------------------

resnet18_model = models.resnet18(weights=None)

resnet18_model.fc = nn.Linear(
    resnet18_model.fc.in_features,
    1
)

# ---------------------------------------
# 2. Load trained checkpoint
# ---------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=True
)

print("\nCHECKPOINT TYPE:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("CHECKPOINT KEYS:")
    print(list(checkpoint.keys())[:15])

# ---------------------------------------
# 3. Identify model weights
# ---------------------------------------

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

    state_dict = checkpoint["model_state_dict"]

elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:

    state_dict = checkpoint["state_dict"]

else:

    state_dict = checkpoint

# ---------------------------------------
# 4. Load weights
# ---------------------------------------

resnet18_model.load_state_dict(
    state_dict,
    strict=True
)

# ---------------------------------------
# 5. Freeze model
# ---------------------------------------

resnet18_model = resnet18_model.to(device)

resnet18_model.eval()

for parameter in resnet18_model.parameters():
    parameter.requires_grad = False

# ---------------------------------------
# 6. Reproduce original preprocessing
# ---------------------------------------

inference_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ---------------------------------------
# 7. Verify model
# ---------------------------------------

total_parameters = sum(
    p.numel()
    for p in resnet18_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in resnet18_model.parameters()
    if p.requires_grad
)

assert trainable_parameters == 0

assert total_parameters == 11177025

print("\nMODEL")
print("Architecture: ResNet18")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

print("\nINFERENCE")
print("Device:", device)
print("Input size: 320 x 320")
print("Study aggregation: Mean image probability")
print("Fixed threshold:", SELECTED_THRESHOLD)

print("\nStep 6A: PASSED")

## Step 6B — Verify ResNet18 Inference Reproducibility

### Objective

Verify that the ResNet18 model loaded in Notebook 09 reproduces the image-level abnormality probabilities generated in Notebook 08.

### Methodology

Five official MURA wrist validation images will be selected for testing.

For each image:

1. Retrieve the original image from the MURA archive.
2. Convert it to RGB.
3. Resize it to 320 × 320 pixels.
4. Apply the original ImageNet normalization.
5. Perform inference using the frozen ResNet18 model.
6. Compare the calculated abnormality probability with the previously saved prediction.

### Acceptance Criterion

The absolute difference between the new and reference probabilities must be less than `1e-5`.

This tolerance accommodates small numerical differences in floating-point calculations.

### Scientific Importance

This verification ensures that the Agentic AI system uses the same classifier as the original validation experiment.

No model weights, preprocessing parameters, aggregation methods, or classification thresholds will be modified.

### Expected Outcome

All five images reproduce their original predictions within the predefined numerical tolerance.

In [ ]:
import io
import numpy as np
import pandas as pd
import zipfile
from PIL import Image

print("NOTEBOOK 09 — STEP 6B: INFERENCE REPRODUCIBILITY")
print("-" * 60)

# ---------------------------------------
# 1. Load original image predictions
# ---------------------------------------

PREDICTIONS_PATH = (
    VALIDATION_DIR /
    "resnet18_official_image_predictions.csv"
)

assert PREDICTIONS_PATH.exists(), (
    "Official image predictions file not found."
)

reference_df = pd.read_csv(PREDICTIONS_PATH)

print("\nREFERENCE DATA")
print("Rows:", len(reference_df))
print("Columns:", reference_df.columns.tolist())

# ---------------------------------------
# 2. Match predictions to image paths
# ---------------------------------------

verification_df = reference_df.merge(
    lookup_df,
    on="image_id",
    how="inner",
    validate="one_to_one"
)

assert len(verification_df) == len(reference_df)

# Select five images reproducibly
test_images = verification_df.sample(
    n=5,
    random_state=42
)

# ---------------------------------------
# 3. Run inference
# ---------------------------------------

verification_results = []

resnet18_model.eval()

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    for row in test_images.itertuples(index=False):

        image_bytes = zf.read(row.image_path)

        with Image.open(io.BytesIO(image_bytes)) as img:
            image = img.convert("RGB")

        image_tensor = (
            inference_transform(image)
            .unsqueeze(0)
            .to(device)
        )

        with torch.inference_mode():

            logit = resnet18_model(image_tensor)

            new_probability = (
                torch.sigmoid(logit)
                .item()
            )

        reference_probability = float(
            row.probability
        )

        absolute_difference = abs(
            new_probability - reference_probability
        )

        verification_results.append({
            "image_id": row.image_id,
            "reference_probability": reference_probability,
            "new_probability": new_probability,
            "absolute_difference": absolute_difference
        })

# ---------------------------------------
# 4. Display comparison
# ---------------------------------------

results_df = pd.DataFrame(
    verification_results
)

print("\nPREDICTION COMPARISON")

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# ---------------------------------------
# 5. Verify reproducibility
# ---------------------------------------

max_difference = (
    results_df["absolute_difference"].max()
)

TOLERANCE = 1e-5

print("\nVERIFICATION")
print("Maximum absolute difference:", max_difference)
print("Tolerance:", TOLERANCE)

assert max_difference < TOLERANCE, (
    "Inference reproducibility test FAILED."
)

print("\nStep 6B: PASSED")

## Step 6C — Implement the Study-Level Classification Tool

### Objective

Develop a reusable `classify_study` function that performs technical validation and frozen ResNet18 inference on a complete MURA wrist study.

### Input

- De-identified study identifier.
- List of image identifiers and image bytes.

### Processing

1. Verify that the study contains at least one image.
2. Apply mandatory technical QC to every image.
3. Block classification if any image fails QC.
4. Convert validated images to RGB.
5. Apply the original ResNet18 preprocessing.
6. Generate individual image-level abnormality probabilities.
7. Calculate the arithmetic mean of image-level probabilities.
8. Apply the frozen study-level classification threshold.

### Output

A structured dictionary containing:

- Study identifier.
- Classification status.
- Number of images.
- Image-level probabilities.
- Study-level mean probability.
- Frozen classification threshold.
- Predicted class.
- Technical QC results.

### Safety Requirements

- The classifier must not modify its weights or threshold.
- Failed QC must not be bypassed.
- No images may be silently excluded from study aggregation.
- The function must not generate specific clinical diagnoses.
- Predictions must remain separate from subsequent agent interpretations.

### Expected Outcome

A deterministic study-level classification tool suitable for integration into the Agentic AI workflow.

In [ ]:
import io
import numpy as np
import torch
from PIL import Image

print("NOTEBOOK 09 — STEP 6C: CLASSIFICATION TOOL")
print("-" * 60)


def classify_study(study_id, images):
    """
    Classify a complete MURA wrist study.

    Parameters
    ----------
    study_id : str
        De-identified study identifier.

    images : list of dict
        Each dictionary contains:
        {
            "image_id": str,
            "image_bytes": bytes
        }

    Returns
    -------
    dict
        Structured study-level classification result.
    """

    result = {
        "study_id": study_id,
        "status": "FAILED",
        "image_count": len(images),
        "image_predictions": [],
        "study_probability": None,
        "threshold": float(SELECTED_THRESHOLD),
        "predicted_class": None,
        "qc_results": [],
        "issues": []
    }

    # ---------------------------------------
    # 1. Validate study input
    # ---------------------------------------

    if len(images) == 0:
        result["issues"].append("empty_study")
        return result

    image_ids = [item["image_id"] for item in images]

    if len(image_ids) != len(set(image_ids)):
        result["issues"].append("duplicate_image_ids")
        return result

    # ---------------------------------------
    # 2. Mandatory QC
    # ---------------------------------------

    for item in images:

        qc_result = technical_qc(
            item["image_bytes"],
            image_id=item["image_id"]
        )

        result["qc_results"].append(qc_result)

    failed_images = [
        item["image_id"]
        for item in result["qc_results"]
        if item["status"] == "FAIL"
    ]

    if failed_images:
        result["issues"].append(
            "qc_failed: " + ", ".join(failed_images)
        )
        return result

    # ---------------------------------------
    # 3. Preprocess images
    # ---------------------------------------

    tensors = []

    for item in images:

        with Image.open(
            io.BytesIO(item["image_bytes"])
        ) as img:

            image = img.convert("RGB")

        tensor = inference_transform(image)

        tensors.append(tensor)

    image_batch = torch.stack(tensors).to(device)

    # ---------------------------------------
    # 4. Frozen ResNet18 inference
    # ---------------------------------------

    resnet18_model.eval()

    with torch.inference_mode():

        logits = resnet18_model(image_batch)

        probabilities = (
            torch.sigmoid(logits)
            .flatten()
            .cpu()
            .numpy()
        )

    # ---------------------------------------
    # 5. Store image-level probabilities
    # ---------------------------------------

    for image_id, probability in zip(
        image_ids,
        probabilities
    ):

        result["image_predictions"].append({
            "image_id": image_id,
            "probability": float(probability)
        })

    # ---------------------------------------
    # 6. Study-level aggregation
    # ---------------------------------------

    study_probability = float(
        np.mean(probabilities, dtype=np.float64)
    )

    predicted_class = int(
        study_probability >= SELECTED_THRESHOLD
    )

    result["study_probability"] = study_probability
    result["predicted_class"] = predicted_class
    result["status"] = "SUCCESS"

    return result


print("classify_study function defined successfully.")
print("\nStep 6C: READY FOR TESTING")

## Step 6D — Study-Level Classification Verification

### Objective

Verify that the newly implemented `classify_study` tool reproduces the study-level predictions generated during the official MURA validation experiment.

### Methodology

1. Select three complete wrist studies from the official validation dataset.
2. Retrieve every image belonging to each selected study.
3. Run the `classify_study` function, including mandatory technical QC.
4. Compare image-level probabilities with the previously saved predictions.
5. Compare mean study-level probabilities and final classifications.

### Acceptance Criteria

- All study images must pass mandatory QC.
- The number of images must match the reference study.
- Maximum absolute difference in image-level probabilities must be below `1e-5`.
- Maximum absolute difference in study-level probabilities must be below `1e-5`.
- Final classifications must match exactly.

### Scientific Importance

This verification confirms that the Agentic AI system preserves the original classification pipeline.

The agent must not modify the model weights, threshold, or aggregation method.

### Expected Outcome

All three studies reproduce the previously saved predictions within the predefined numerical tolerance.

In [ ]:
import numpy as np
import pandas as pd
import zipfile

print("NOTEBOOK 09 — STEP 6D: STUDY REPRODUCIBILITY")
print("-" * 60)

# Load reference study predictions
STUDY_PREDICTIONS_PATH = (
    VALIDATION_DIR /
    "resnet18_official_study_predictions.csv"
)

reference_studies = pd.read_csv(
    STUDY_PREDICTIONS_PATH
)

print("\nREFERENCE COLUMNS")
print(reference_studies.columns.tolist())

# Select three complete studies
# Use a fixed random seed for reproducibility.
selected_studies = (
    reference_studies
    .sample(n=3, random_state=42)
)

verification_records = []

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    for reference in selected_studies.itertuples(index=False):

        study_id = reference.study_id

        study_images_df = (
            verification_df[
                verification_df["study_id"] == study_id
            ]
            .sort_values("image_id")
        )

        images = []

        for row in study_images_df.itertuples(index=False):

            images.append({
                "image_id": row.image_id,
                "image_bytes": zf.read(row.image_path)
            })

        # Run our new classification tool
        result = classify_study(
            study_id=study_id,
            images=images
        )

        assert result["status"] == "SUCCESS", (
            f"Classification failed for {study_id}: "
            f"{result['issues']}"
        )

        # Compare image-level predictions
        reference_image_probs = dict(
            zip(
                study_images_df["image_id"],
                study_images_df["probability"]
            )
        )

        image_differences = [
            abs(
                item["probability"]
                - reference_image_probs[item["image_id"]]
            )
            for item in result["image_predictions"]
        ]

        max_image_difference = max(image_differences)

        # Compare study-level results
        reference_study_probability = float(
            reference.mean_probability
        )

        study_difference = abs(
            result["study_probability"]
            - reference_study_probability
        )

        reference_class = int(
            reference.predicted_class
        )

        verification_records.append({
            "study_id": study_id,
            "image_count": result["image_count"],
            "max_image_difference": max_image_difference,
            "study_difference": study_difference,
            "reference_class": reference_class,
            "new_class": result["predicted_class"],
            "qc_passed": all(
                q["status"] != "FAIL"
                for q in result["qc_results"]
            )
        })

results_6d = pd.DataFrame(
    verification_records
)

print("\nSTUDY COMPARISON")
print(
    results_6d.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# Verify all acceptance criteria
assert (
    results_6d["max_image_difference"] < 1e-5
).all()

assert (
    results_6d["study_difference"] < 1e-5
).all()

assert (
    results_6d["reference_class"]
    == results_6d["new_class"]
).all()

assert results_6d["qc_passed"].all()

print("\nStep 6D: PASSED")

All three studies reproduced the original Notebook 08 predictions within our predefined tolerance.
**Tool 2: classify_study	Uses ResNet18 to predict normal versus abnormal for a complete wrist study	✅ Implemented and reproducibility verified.**

## Step 7 — Anatomical Scope Validation

### Objective

Develop a mandatory anatomical scope validator that determines whether an uploaded radiograph belongs to the supported anatomical region: the wrist.

### Scientific Rationale

The existing ResNet18 classifier was trained exclusively for normal-versus-abnormal classification of wrist radiographs.

It was not trained to identify anatomical regions.

Therefore, a radiograph from another anatomical region could receive a misleading wrist abnormality prediction.

### Proposed Solution

Develop an independent anatomical recognition component using MURA's seven anatomical categories.

The component will distinguish supported wrist images from known unsupported anatomical regions and investigate methods for rejecting unfamiliar inputs.

### Safety Policy

The anatomical validator must execute before the wrist abnormality classifier.

Possible outcomes:

- **SUPPORTED_WRIST:** Sufficient evidence that the input belongs to the supported anatomical region.
- **UNSUPPORTED_ANATOMY:** Evidence that the image belongs to a different anatomical region.
- **UNCERTAIN_ANATOMY:** Insufficient confidence to establish anatomical compatibility.

Only `SUPPORTED_WRIST` permits wrist abnormality classification.

The agent cannot override anatomical scope decisions.

### Step 7A — Dataset Exploration

Examine the distribution of images, studies, and patients across MURA's seven anatomical categories.

The analysis will use the training partition for model-development planning.

The official validation partition will remain separate from training and threshold calibration.

### Expected Outcome

A dataset summary that supports selection of an appropriate anatomical recognition model and evaluation strategy.

In [ ]:
import pandas as pd
import zipfile

print("NOTEBOOK 09 — STEP 7A: ANATOMICAL DATA EXPLORATION")
print("-" * 60)

records = []

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    for path in zf.namelist():

        parts = path.split("/")

        # Expected:
        # train/XR_WRIST/patient.../study.../image.png

        if (
            len(parts) == 5
            and parts[0] == "train"
            and parts[1].startswith("XR_")
            and parts[4].lower().endswith(".png")
        ):

            records.append({
                "anatomy": parts[1],
                "patient_id": parts[2],
                "study_id": "/".join(parts[:4]),
                "image_path": path
            })

anatomy_df = pd.DataFrame(records)

assert not anatomy_df.empty, (
    "No training images found. Check ZIP structure."
)

# Count images, studies, and patients
anatomy_summary = (
    anatomy_df.groupby("anatomy")
    .agg(
        images=("image_path", "count"),
        studies=("study_id", "nunique"),
        patients=("patient_id", "nunique")
    )
    .sort_values("images", ascending=False)
)

print("\nMURA TRAINING DATASET")
print(anatomy_summary.to_string())

print("\nTOTALS")
print("Anatomical categories:", anatomy_df["anatomy"].nunique())
print("Images:", len(anatomy_df))
print("Studies:", anatomy_df["study_id"].nunique())

print("\nStep 7A: PASSED")

In [ ]:
import zipfile

print("MURA — AVAILABLE METADATA FILES")
print("-" * 50)

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    metadata_files = [
        name for name in zf.namelist()
        if name.lower().endswith(
            (".csv", ".json", ".txt", ".dcm")
        )
    ]

    for name in sorted(metadata_files):
        print(name)

    print("\nTOTAL:", len(metadata_files))

In [ ]:
import pandas as pd
import zipfile

print("NOTEBOOK 09 — STEP 7B: MURA METADATA INSPECTION")
print("-" * 60)

metadata_files = [
    "train_image_paths.csv",
    "train_labeled_studies.csv",
    "valid_image_paths.csv",
    "valid_labeled_studies.csv"
]

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    for filename in metadata_files:

        print(f"\n{'=' * 60}")
        print(f"FILE: {filename}")
        print("=" * 60)

        with zf.open(filename) as file:
            df = pd.read_csv(file, header=None)

        print(f"Rows: {len(df)}")
        print(f"Columns: {len(df.columns)}")

        display(df.head(5))

print("\nStep 7B: COMPLETED")

## Step 7C — Metadata-Based Anatomical Scope Validation

### Objective

Implement a deterministic safety tool that verifies whether a radiographic study is declared to represent the wrist before allowing it to enter the frozen ResNet18 abnormality classifier.

### Method

For the MURA dataset, anatomical information will be extracted from the image or study path.

The validator will return one of three outcomes:

- **SUPPORTED_WRIST:** The metadata identifies the anatomical region as XR_WRIST.
- **UNSUPPORTED_ANATOMY:** The metadata identifies another anatomical region.
- **UNCERTAIN_ANATOMY:** Anatomical information is missing, malformed, or inconsistent.

Only studies with a SUPPORTED_WRIST result may proceed to wrist abnormality classification.

### Safety Limitations

This validator evaluates anatomical metadata rather than image content.

It cannot identify an incorrectly labeled radiograph.

Future extensions may incorporate DICOM metadata extraction and independent image-based anatomical verification.

The anatomical scope decision must be enforced before classification and cannot be overridden by the AI agent.

In [ ]:
from pathlib import PurePosixPath

SUPPORTED_ANATOMY = "XR_WRIST"

KNOWN_MURA_ANATOMIES = {
    "XR_WRIST",
    "XR_HAND",
    "XR_FINGER",
    "XR_FOREARM",
    "XR_ELBOW",
    "XR_HUMERUS",
    "XR_SHOULDER",
}


def anatomical_scope_validator(image_paths):
    """
    Validate declared anatomy using MURA image paths.

    Parameters
    ----------
    image_paths : list[str]
        Paths belonging to one radiographic study.

    Returns
    -------
    dict
        Anatomical scope validation result.
    """

    if not image_paths:
        return {
            "status": "UNCERTAIN_ANATOMY",
            "anatomical_region": None,
            "metadata_source": "MURA_PATH",
            "issues": ["No image paths provided"],
        }

    detected_regions = []
    issues = []

    for path in image_paths:

        if not isinstance(path, str) or not path.strip():
            issues.append("Missing or invalid image path")
            continue

        parts = PurePosixPath(
            path.replace("\\", "/")
        ).parts

        # Expected structure:
        # .../train/XR_WRIST/patient.../study.../image...
        # .../valid/XR_WRIST/patient.../study.../image...

        matches = [
            parts[i + 1]
            for i, part in enumerate(parts[:-1])
            if part in {"train", "valid"}
        ]

        if len(matches) != 1:
            issues.append(f"Malformed MURA path: {path}")
            continue

        detected_regions.append(matches[0])

    unique_regions = set(detected_regions)

    if issues or len(unique_regions) != 1:
        return {
            "status": "UNCERTAIN_ANATOMY",
            "anatomical_region": None,
            "metadata_source": "MURA_PATH",
            "issues": issues or ["Conflicting anatomical regions"],
        }

    region = unique_regions.pop()

    if region == SUPPORTED_ANATOMY:
        status = "SUPPORTED_WRIST"
    elif region in KNOWN_MURA_ANATOMIES:
        status = "UNSUPPORTED_ANATOMY"
    else:
        status = "UNCERTAIN_ANATOMY"
        issues.append(f"Unrecognized anatomical region: {region}")

    return {
        "status": status,
        "anatomical_region": region,
        "metadata_source": "MURA_PATH",
        "anatomy_image_verified": False,
        "issues": issues,
    }

In [ ]:
print("NOTEBOOK 09 — STEP 7C: ANATOMICAL SCOPE TESTS")
print("-" * 60)

test_cases = {
    "Wrist": [
        "MURA-v1.1/valid/XR_WRIST/patient11185/study1_positive/image1.png"
    ],
    "Shoulder": [
        "MURA-v1.1/train/XR_SHOULDER/patient00001/study1_positive/image1.png"
    ],
    "Hand": [
        "MURA-v1.1/train/XR_HAND/patient00001/study1_positive/image1.png"
    ],
    "Unknown anatomy": [
        "MURA-v1.1/train/XR_KNEE/patient00001/study1_positive/image1.png"
    ],
    "Mixed anatomy": [
        "MURA-v1.1/valid/XR_WRIST/patient11185/study1_positive/image1.png",
        "MURA-v1.1/train/XR_HAND/patient00001/study1_positive/image1.png"
    ],
    "Missing metadata": [],
}

expected = {
    "Wrist": "SUPPORTED_WRIST",
    "Shoulder": "UNSUPPORTED_ANATOMY",
    "Hand": "UNSUPPORTED_ANATOMY",
    "Unknown anatomy": "UNCERTAIN_ANATOMY",
    "Mixed anatomy": "UNCERTAIN_ANATOMY",
    "Missing metadata": "UNCERTAIN_ANATOMY",
}

for name, paths in test_cases.items():

    result = anatomical_scope_validator(paths)

    print(
        f"{name:20s} → {result['status']}"
    )

    assert result["status"] == expected[name]

print("\nStep 7C: PASSED")

## Step 7D — Integration of Anatomical Scope Validation

### Objective

Integrate the metadata-based anatomical scope validator into the existing classification workflow.

### Workflow

Each radiographic study must pass two mandatory checks before abnormality classification:

1. **Technical QC:** Verify that the image data are readable and satisfy the predefined technical criteria.
2. **Anatomical Scope Validation:** Verify that the study metadata identifies the anatomical region as XR_WRIST.

Only studies passing both checks are eligible for classification using the frozen ResNet18 model.

### Safety Rules

- Unsupported anatomical regions must not be classified.
- Missing or conflicting anatomical metadata must trigger abstention.
- The AI agent cannot override either mandatory check.
- The anatomical validator assesses metadata, not the actual anatomy visible in the image.
- The frozen model weights, classification threshold, and study-level aggregation method remain unchanged.

### Expected Outcome

A guarded classification function that returns either:

- **SUCCESS:** The study passed both mandatory checks and was classified.
- **BLOCKED:** The study failed a mandatory check and was not classified.

All decisions must include the reason for acceptance or rejection.

In [ ]:
def guarded_classify_study(study_id, images, image_paths):
    """
    Guarded wrist abnormality classification.

    Mandatory order:
        1. Technical QC
        2. Anatomical scope validation
        3. Frozen ResNet18 classification

    Parameters
    ----------
    study_id : str
        Study identifier.

    images : list[dict]
        Each dictionary contains:
            image_id
            image_bytes

    image_paths : list[str]
        MURA paths corresponding to the supplied images.
    """

    # ---------------------------------------
    # 1. Verify study input
    # ---------------------------------------

    if not images or len(images) != len(image_paths):
        return {
            "study_id": study_id,
            "status": "BLOCKED",
            "reason": "Missing images or mismatched metadata",
            "classification_performed": False,
        }

    # ---------------------------------------
    # 2. Mandatory technical QC
    # ---------------------------------------

    qc_results = [
        technical_qc(
            image["image_bytes"],
            image_id=image["image_id"]
        )
        for image in images
    ]

    failed_qc = [
        result
        for result in qc_results
        if result["status"] == "FAIL"
    ]

    if failed_qc:
        return {
            "study_id": study_id,
            "status": "BLOCKED",
            "reason": "Technical QC failure",
            "classification_performed": False,
            "qc_results": qc_results,
        }

    # ---------------------------------------
    # 3. Mandatory anatomical scope check
    # ---------------------------------------

    anatomy_result = anatomical_scope_validator(
        image_paths
    )

    if anatomy_result["status"] != "SUPPORTED_WRIST":
        return {
            "study_id": study_id,
            "status": "BLOCKED",
            "reason": anatomy_result["status"],
            "classification_performed": False,
            "anatomy_result": anatomy_result,
            "qc_results": qc_results,
        }

    # ---------------------------------------
    # 4. Frozen ResNet18 classification
    # ---------------------------------------

    classification = classify_study(
        study_id=study_id,
        images=images
    )

    if classification["status"] != "SUCCESS":
        return {
            "study_id": study_id,
            "status": "BLOCKED",
            "reason": "Classification tool did not succeed",
            "classification_performed": False,
            "classification_result": classification,
            "anatomy_result": anatomy_result,
            "qc_results": qc_results,
        }

    return {
        "study_id": study_id,
        "status": "SUCCESS",
        "classification_performed": True,
        "anatomy_result": anatomy_result,
        "classification_result": classification,
        "qc_results": qc_results,
    }


print("Step 7D: Guarded classifier defined successfully.")

## Step 7E — Anatomical Guardrail Integration Testing

### Objective

Verify that the guarded classification workflow enforces mandatory technical QC and anatomical scope validation before invoking the frozen ResNet18 classifier.

### Test Cases

1. **Supported wrist:** A valid wrist study should proceed to classification.
2. **Unsupported anatomy:** A study declared as XR_SHOULDER should be blocked.
3. **Missing metadata:** A study without anatomical metadata should be blocked.

### Evaluation Criteria

- Unsupported or uncertain anatomy must never trigger ResNet18 inference.
- Supported wrist studies must remain eligible for classification.
- Rejected studies must return an explicit reason.
- The existing model weights, threshold, and aggregation method must remain unchanged.

### Limitation

The validator checks anatomical metadata rather than image content. Incorrectly labeled radiographs cannot be identified by this guardrail alone.

In [ ]:
print("STEP 7E — CHECK TEST IMAGES")
print("-" * 40)

print("Type:", type(test_images))
print("Number of images:", len(test_images))

if len(test_images) > 0:
    # Using .iloc[0] because test_images is a pandas DataFrame
    first_row = test_images.iloc[0]
    print("First item type:", type(first_row))
    print("First item index/columns:", list(first_row.index))

In [ ]:
print("STEP 7E — RESTORE GUARDRAIL TEST IMAGE")
print("-" * 50)

from pathlib import Path
import zipfile

# Select one verified wrist image
wrist_row = lookup_df.iloc[0]

guardrail_image_id = str(wrist_row["image_id"])
guardrail_image_path = str(wrist_row["image_path"])

# Read the original image bytes from the MURA archive
with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    archive_members = zf.namelist()

    matches = [
        name for name in archive_members
        if name.endswith(guardrail_image_path.lstrip("/"))
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one matching image, found {len(matches)}"
        )

    guardrail_image_bytes = zf.read(matches[0])

# Prepare the input expected by the classifier
guardrail_test_images = [
    {
        "image_id": guardrail_image_id,
        "image_bytes": guardrail_image_bytes
    }
]

print("Image ID:", guardrail_image_id)
print("Image path:", guardrail_image_path)
print("Number of images:", len(guardrail_test_images))
print("Image bytes:", len(guardrail_image_bytes))

assert len(guardrail_test_images) == 1
assert len(guardrail_image_bytes) > 0

print("\nGuardrail test image restored successfully.")

## Step 7 — Anatomical Scope Validation: Summary

### Objective

Develop and integrate a mandatory anatomical scope validator to prevent radiographs with unsupported or uncertain anatomical metadata from entering the wrist abnormality classification workflow.

### Method

A deterministic metadata-based validator was implemented using the anatomical region encoded in MURA image paths.

The validator distinguishes between:

- **SUPPORTED_WRIST:** Metadata identifies the wrist as the anatomical region.
- **UNSUPPORTED_ANATOMY:** Metadata identifies another recognized anatomical region.
- **UNCERTAIN_ANATOMY:** Anatomical metadata is missing, malformed, inconsistent, or unrecognized.

Only studies classified as SUPPORTED_WRIST are eligible for abnormality classification.

### Validation Results

Six unit tests verified the anatomical validator's handling of supported, unsupported, and uncertain metadata.

Three integration tests evaluated the guarded classification workflow:

| Test | Result | Classifier invoked |
|---|---|---|
| Supported wrist | SUCCESS | Yes |
| Unsupported shoulder | BLOCKED | No |
| Missing anatomical metadata | BLOCKED | No |

All tests passed.

### Integration

The guarded workflow executes the following mandatory sequence:

**Technical QC → Anatomical Scope Validation → Frozen ResNet18 Classification**

The original ResNet18 weights, decision threshold, and study-level aggregation method were not modified.

### Limitations

The anatomical validator evaluates metadata rather than image content. It cannot independently verify that the anatomy visible in a radiograph matches its declared anatomical region.

The integration tests demonstrate blocking behavior through the guarded workflow. They do not establish that all possible direct calls to the underlying classifier are protected.

The MURA dataset contains PNG images rather than original DICOM files. Future development may extend the validator to extract anatomical metadata directly from DICOM headers.

### Conclusion

The anatomical scope validator successfully prevented unsupported and uncertain anatomical metadata from reaching the frozen wrist abnormality classifier in the tested workflow.

**Step 7: PASSED.**

## Step 7F — Update the Agent Specification

### Objective

Update the original agent specification to incorporate the anatomical scope validator developed and tested in Step 7.

### Changes

The agent now has eight tools, including three mandatory safeguards or processing stages:

1. **technical_qc** — Mandatory image integrity assessment.
2. **anatomical_scope_validator** — Mandatory metadata-based anatomical eligibility assessment.
3. **classify_study** — Mandatory classification for eligible studies only.

The remaining tools support view-consistency analysis, uncertainty assessment, explainability, metadata retrieval, and reporting.

### Safety Requirements

- Technical QC must precede anatomical scope validation.
- Anatomical scope validation must precede abnormality classification.
- Unsupported or uncertain anatomical metadata must block classification.
- The AI agent must not bypass mandatory safeguards.
- The agent must access wrist classification only through the guarded workflow.
- Frozen model weights, decision threshold, and study-level aggregation must remain unchanged.

### Expected Outcome

An updated, auditable agent specification containing eight registered tools and the revised execution requirements.

In [ ]:
import json
from pathlib import Path

print("NOTEBOOK 09 — STEP 7F: AGENT SPECIFICATION UPDATE")
print("-" * 60)

spec_path = AGENT_DIR / "config" / "mura_agent_specification.json"

assert spec_path.exists(), "Original agent specification not found."

with open(spec_path, "r", encoding="utf-8") as f:
    agent_spec = json.load(f)

# Inspect the existing structure before modifying it.
print("\nExisting specification keys:")
print(list(agent_spec.keys()))

print("\nExisting tools structure:")
print(type(agent_spec.get("tools")).__name__)

print("\nExisting tools:")
print(json.dumps(agent_spec.get("tools"), indent=2))

print("\nStep 7F — Inspection completed.")

In [ ]:
import json
from datetime import datetime, timezone

print("NOTEBOOK 09 — STEP 7F: SAVE UPDATED AGENT SPECIFICATION")
print("-" * 60)

# ---------------------------------------
# 1. Register anatomical scope validator
# ---------------------------------------

agent_spec["tools"]["anatomical_scope_validator"] = {
    "execution": "mandatory_after_qc",
    "description": (
        "Metadata-based anatomical scope validation. "
        "Only studies declared as XR_WRIST may proceed "
        "to abnormality classification."
    )
}

# ---------------------------------------
# 2. Update classification requirements
# ---------------------------------------

agent_spec["tools"]["classify_study"]["execution"] = (
    "mandatory_after_qc_and_anatomical_validation"
)

agent_spec["tools"]["classify_study"]["description"] = (
    "Frozen ResNet18 wrist abnormality inference. "
    "Accessible to the agent only through "
    "guarded_classify_study."
)

# ---------------------------------------
# 3. Define mandatory execution sequence
# ---------------------------------------

agent_spec["mandatory_workflow"] = [
    "technical_qc",
    "anatomical_scope_validator",
    "classify_study",
    "view_consistency",
    "report_generator"
]

# ---------------------------------------
# 4. Add safety constraints
# ---------------------------------------

new_constraints = [
    "Anatomical scope validation must precede wrist classification.",
    "Unsupported or uncertain anatomical metadata must block classification.",
    "The agent must access wrist classification only through guarded_classify_study.",
    "The agent cannot override mandatory technical or anatomical safeguards."
]

# Preserve the original safety_constraints structure.
existing_constraints = agent_spec["safety_constraints"]

if isinstance(existing_constraints, list):
    for constraint in new_constraints:
        if constraint not in existing_constraints:
            existing_constraints.append(constraint)

elif isinstance(existing_constraints, dict):
    existing_constraints["anatomical_scope_guardrails"] = (
        new_constraints
    )

else:
    raise TypeError(
        "Unexpected safety_constraints structure: "
        f"{type(existing_constraints).__name__}"
    )

# ---------------------------------------
# 5. Update specification metadata
# ---------------------------------------

agent_spec["version"] = "1.1"

agent_spec["updated_at"] = (
    datetime.now(timezone.utc).isoformat()
)

# ---------------------------------------
# 6. Save updated specification
# ---------------------------------------

with open(spec_path, "w", encoding="utf-8") as f:
    json.dump(agent_spec, f, indent=2)

# ---------------------------------------
# 7. Verify saved specification
# ---------------------------------------

with open(spec_path, "r", encoding="utf-8") as f:
    saved_spec = json.load(f)

assert len(saved_spec["tools"]) == 8

assert (
    saved_spec["tools"]["anatomical_scope_validator"]["execution"]
    == "mandatory_after_qc"
)

assert (
    saved_spec["tools"]["classify_study"]["execution"]
    == "mandatory_after_qc_and_anatomical_validation"
)

print("\nREGISTERED TOOLS:")
for name, details in saved_spec["tools"].items():
    print(f"{name:30s} {details['execution']}")

print("\nMANDATORY WORKFLOW:")
print(" → ".join(saved_spec["mandatory_workflow"]))

print("\nSpecification version:", saved_spec["version"])
print("Total tools:", len(saved_spec["tools"]))
print("Saved to:", spec_path)

print("\nStep 7F: PASSED")

## Step 8A — View Consistency Analysis

### Objective

Develop a deterministic tool that measures disagreement between image-level predictions within a radiographic study.

### Rationale

The frozen ResNet18 model produces an abnormality probability for each image.

Study-level classification uses the arithmetic mean of image probabilities and the frozen decision threshold.

However, individual images may produce substantially different probabilities.

Such disagreement may indicate that the study requires additional analysis.

### Method

For each study, calculate:

- **Minimum probability:** Lowest image-level abnormality probability.
- **Maximum probability:** Highest image-level abnormality probability.
- **Probability range:** Maximum minus minimum.
- **Standard deviation:** Dispersion of image-level probabilities.
- **Threshold disagreement:** Whether some images are above and others below the frozen classification threshold.

### Interpretation

Threshold disagreement identifies studies in which individual image predictions do not agree on the binary classification.

Probability range and standard deviation provide additional descriptive measures of variability.

These metrics do not independently establish clinical uncertainty, image quality, or the presence of an abnormality.

### Safety Constraints

- The frozen ResNet18 model remains unchanged.
- The selected classification threshold remains unchanged.
- Study-level aggregation remains the arithmetic mean.
- View-consistency results do not override the study-level classification.
- Human escalation criteria will be developed and evaluated separately.

### Expected Outcome

A reproducible view-consistency tool that provides structured evidence for subsequent agentic decision-making.

In [ ]:
import numpy as np

print("NOTEBOOK 09 — STEP 8A: VIEW CONSISTENCY TOOL")
print("-" * 60)


def view_consistency(image_predictions, threshold=SELECTED_THRESHOLD):
    """
    Analyze disagreement between image-level predictions.

    Parameters
    ----------
    image_predictions : list[dict]
        Each dictionary contains:
            image_id
            probability

    threshold : float
        Frozen classification threshold.

    Returns
    -------
    dict
        View-consistency metrics.
    """

    if not image_predictions:
        return {
            "status": "INSUFFICIENT_DATA",
            "reason": "No image predictions provided"
        }

    probabilities = np.array(
        [
            item["probability"]
            for item in image_predictions
        ],
        dtype=np.float64
    )

    if not np.all(np.isfinite(probabilities)):
        return {
            "status": "INVALID_INPUT",
            "reason": "Non-finite probabilities detected"
        }

    if np.any((probabilities < 0) | (probabilities > 1)):
        return {
            "status": "INVALID_INPUT",
            "reason": "Probabilities must be between 0 and 1"
        }

    image_classes = (
        probabilities >= threshold
    ).astype(int)

    threshold_disagreement = (
        len(np.unique(image_classes)) > 1
    )

    return {
        "status": "SUCCESS",
        "image_count": len(probabilities),
        "min_probability": float(np.min(probabilities)),
        "max_probability": float(np.max(probabilities)),
        "mean_probability": float(np.mean(probabilities)),
        "probability_range": float(np.ptp(probabilities)),
        "probability_std": float(np.std(probabilities)),
        "threshold_disagreement": bool(threshold_disagreement),
        "image_classes": image_classes.tolist(),
        "threshold": float(threshold)
    }


print("Step 8A: View consistency tool defined successfully.")

## Step 8B — View Consistency Unit Tests

### Objective

Verify that the view-consistency tool correctly calculates probability dispersion and identifies disagreement between image-level predictions.

### Test Design

Five synthetic test cases will be evaluated:

1. All image probabilities below the frozen threshold.
2. All image probabilities above the frozen threshold.
3. Mixed probabilities above and below the frozen threshold.
4. A study containing only one image.
5. An invalid probability outside the interval [0, 1].

### Acceptance Criteria

- Probability range and standard deviation are calculated correctly.
- Threshold disagreement is detected only when image-level classifications differ.
- Single-image studies do not produce artificial disagreement.
- Invalid probabilities are rejected.
- The frozen classification threshold remains unchanged.

### Interpretation

These tests verify the mathematical and logical behavior of the tool.

They do not establish whether prediction disagreement reliably identifies clinically abnormal studies or improves diagnostic performance.

In [ ]:
print("NOTEBOOK 09 — STEP 8B: VIEW CONSISTENCY UNIT TESTS")
print("-" * 60)

threshold = SELECTED_THRESHOLD

# ---------------------------------------
# TEST 1 — All below threshold
# ---------------------------------------

test1 = [
    {"image_id": "img1", "probability": 0.01},
    {"image_id": "img2", "probability": 0.02},
    {"image_id": "img3", "probability": 0.03},
]

result1 = view_consistency(test1)

assert result1["status"] == "SUCCESS"
assert result1["threshold_disagreement"] is False
assert result1["image_classes"] == [0, 0, 0]

print("TEST 1 — All below threshold: PASSED")


# ---------------------------------------
# TEST 2 — All above threshold
# ---------------------------------------

test2 = [
    {"image_id": "img1", "probability": 0.20},
    {"image_id": "img2", "probability": 0.40},
    {"image_id": "img3", "probability": 0.80},
]

result2 = view_consistency(test2)

assert result2["status"] == "SUCCESS"
assert result2["threshold_disagreement"] is False
assert result2["image_classes"] == [1, 1, 1]

print("TEST 2 — All above threshold: PASSED")


# ---------------------------------------
# TEST 3 — Threshold disagreement
# ---------------------------------------

test3 = [
    {"image_id": "img1", "probability": 0.01},
    {"image_id": "img2", "probability": 0.03},
    {"image_id": "img3", "probability": 0.30},
]

result3 = view_consistency(test3)

assert result3["status"] == "SUCCESS"
assert result3["threshold_disagreement"] is True
assert result3["image_classes"] == [0, 0, 1]

assert np.isclose(
    result3["probability_range"],
    0.29
)

assert np.isclose(
    result3["probability_std"],
    np.std([0.01, 0.03, 0.30])
)

print("TEST 3 — Threshold disagreement: PASSED")


# ---------------------------------------
# TEST 4 — Single-image study
# ---------------------------------------

test4 = [
    {"image_id": "img1", "probability": 0.10}
]

result4 = view_consistency(test4)

assert result4["status"] == "SUCCESS"
assert result4["threshold_disagreement"] is False
assert result4["probability_range"] == 0.0
assert result4["probability_std"] == 0.0

print("TEST 4 — Single-image study: PASSED")


# ---------------------------------------
# TEST 5 — Invalid probability
# ---------------------------------------

test5 = [
    {"image_id": "img1", "probability": 1.20}
]

result5 = view_consistency(test5)

assert result5["status"] == "INVALID_INPUT"

print("TEST 5 — Invalid probability: PASSED")


print("\nSELECTED THRESHOLD:", threshold)
print("\nStep 8B: PASSED")

## Step 8C — View Consistency Analysis of Official MURA Wrist Studies

### Objective

Apply the validated view-consistency tool to the frozen ResNet18 predictions from the official MURA wrist validation set.

### Dataset

The analysis uses the previously generated image-level predictions from Notebook 08.

- 659 wrist radiographs
- 237 radiographic studies
- Frozen ResNet18 model
- Frozen decision threshold: 0.0600893441587686

### Methods

For each study, calculate:

1. Number of radiographic images.
2. Minimum and maximum image-level abnormality probabilities.
3. Mean abnormality probability.
4. Probability range.
5. Probability standard deviation.
6. Presence or absence of threshold disagreement.

### Evaluation

The analysis will quantify:

- The proportion of studies exhibiting threshold disagreement.
- Differences in probability dispersion between agreeing and disagreeing studies.
- The relationship between threshold disagreement and study-level classification.

A subsequent analysis will examine disagreement among the previously identified false-negative studies.

### Scientific Considerations

This is a descriptive, retrospective analysis of the official MURA validation dataset.

The validation data will not be used to change model weights, the classification threshold, or study-level aggregation.

Any escalation policy suggested by these findings will require separate development and independent evaluation.

In [ ]:
import pandas as pd
import numpy as np

print("NOTEBOOK 09 — STEP 8C: OFFICIAL VALIDATION VIEW CONSISTENCY")
print("-" * 65)

# ---------------------------------------
# 1. Load frozen image-level predictions
# ---------------------------------------

predictions_df = pd.read_csv(
    VALIDATION_DIR / "resnet18_official_image_predictions.csv"
)

print("\nImages:", len(predictions_df))
print("Studies:", predictions_df["study_id"].nunique())

assert len(predictions_df) == 659
assert predictions_df["study_id"].nunique() == 237

# ---------------------------------------
# 2. Calculate view-consistency metrics
# ---------------------------------------

study_results = []

for study_id, group in predictions_df.groupby("study_id"):

    image_predictions = [
        {
            "image_id": str(row["image_id"]),
            "probability": float(row["probability"])
        }
        for _, row in group.iterrows()
    ]

    result = view_consistency(
        image_predictions,
        threshold=SELECTED_THRESHOLD
    )

    assert result["status"] == "SUCCESS"

    study_results.append({
        "study_id": study_id,
        "image_count": result["image_count"],
        "min_probability": result["min_probability"],
        "max_probability": result["max_probability"],
        "mean_probability": result["mean_probability"],
        "probability_range": result["probability_range"],
        "probability_std": result["probability_std"],
        "threshold_disagreement": result["threshold_disagreement"],
        "predicted_class": int(
            result["mean_probability"] >= SELECTED_THRESHOLD
        )
    })

consistency_df = pd.DataFrame(study_results)

# ---------------------------------------
# 3. Verify study-level reproducibility
# ---------------------------------------

reference_studies = pd.read_csv(
    VALIDATION_DIR / "resnet18_official_study_predictions.csv"
)

verification = consistency_df.merge(
    reference_studies[
        ["study_id", "mean_probability", "predicted_class"]
    ],
    on="study_id",
    suffixes=("_new", "_reference"),
    validate="one_to_one"
)

assert len(verification) == 237

assert (
    verification["predicted_class_new"]
    == verification["predicted_class_reference"]
).all()

max_difference = np.max(
    np.abs(
        verification["mean_probability_new"]
        - verification["mean_probability_reference"]
    )
)

assert max_difference < 1e-5

# ---------------------------------------
# 4. Summary statistics
# ---------------------------------------

n_studies = len(consistency_df)

n_disagreement = int(
    consistency_df["threshold_disagreement"].sum()
)

print("\nVIEW CONSISTENCY SUMMARY")
print(f"Total studies: {n_studies}")
print(f"Studies with threshold disagreement: {n_disagreement}")
print(
    "Percentage with disagreement: "
    f"{100 * n_disagreement / n_studies:.2f}%"
)

print("\nPROBABILITY DISPERSION BY GROUP")

summary = (
    consistency_df.groupby("threshold_disagreement")
    .agg(
        studies=("study_id", "count"),
        mean_probability_range=("probability_range", "mean"),
        median_probability_range=("probability_range", "median"),
        mean_probability_std=("probability_std", "mean")
    )
)

print(summary.to_string())

print("\nDISAGREEMENT BY PREDICTED CLASS")

print(
    pd.crosstab(
        consistency_df["predicted_class"],
        consistency_df["threshold_disagreement"],
        rownames=["Predicted class"],
        colnames=["Threshold disagreement"]
    ).to_string()
)

# ---------------------------------------
# 5. Save results
# ---------------------------------------

output_path = (
    AGENT_DIR
    / "evaluation"
    / "mura_official_view_consistency.csv"
)

consistency_df.to_csv(
    output_path,
    index=False
)

print("\nSaved to:", output_path)
print("Maximum study probability difference:", max_difference)

print("\nStep 8C: PASSED")

## Step 8D — Relationship Between View Disagreement and Classification Errors

### Objective

Investigate the relationship between image-level threshold disagreement and study-level classification outcomes in the official MURA wrist validation set.

### Background

The frozen ResNet18 classifier produced nine false-negative study classifications during the held-out evaluation in Notebook 08.

Previous analysis identified five false-negative studies containing at least one image-level probability above the frozen decision threshold.

Step 8C demonstrated that 72 of 237 wrist studies (30.38%) exhibit threshold disagreement.

### Methods

Combine the view-consistency results with the original study-level ground-truth labels.

For each study, determine its classification outcome:

- True positive (TP)
- True negative (TN)
- False positive (FP)
- False negative (FN)

Calculate the frequency of threshold disagreement within each outcome category.

### Evaluation

Particular attention will be given to:

1. The proportion of false-negative studies exhibiting disagreement.
2. The proportion of correctly classified negative studies exhibiting disagreement.
3. Whether disagreement also occurs frequently in false-positive and true-positive studies.

### Scientific Limitations

This is a retrospective exploratory analysis of the official validation dataset.

The results must not be used to modify the frozen classifier, decision threshold, or aggregation method.

Any agent escalation rule motivated by these observations must be evaluated separately to establish its effectiveness, false-alert burden, and generalizability.

In [ ]:
print("NOTEBOOK 09 — STEP 8D: GROUND-TRUTH INSPECTION")
print("-" * 60)

print("\nREFERENCE STUDIES — COLUMNS")
print(reference_studies.columns.tolist())

print("\nREFERENCE STUDIES — FIRST 3 ROWS")
display(reference_studies.head(3))

print("\nVALIDATION DIRECTORY — CSV FILES")

for path in sorted(VALIDATION_DIR.glob("*.csv")):
    print(path.name)

print("\nStep 8D: Ground-truth inspection completed.")

Verify study-level ground truth

In [ ]:
import pandas as pd

print("NOTEBOOK 09 — STEP 8D: VERIFY STUDY-LEVEL GROUND TRUTH")
print("-" * 65)

# Load the original Notebook 08 evaluation
study_eval_df = pd.read_csv(
    VALIDATION_DIR / "resnet18_official_study_evaluation.csv"
)

print("\nDATASET SHAPE")
print(study_eval_df.shape)

print("\nAVAILABLE COLUMNS")
print(study_eval_df.columns.tolist())

print("\nFIRST FIVE ROWS")
display(study_eval_df.head())

print("\nMISSING VALUES")
print(study_eval_df.isna().sum().to_string())

print("\nStep 8D: Ground-truth file loaded successfully.")

We will combine the ground-truth labels from Notebook 08 with the view-consistency results from Step 8C.

The analysis will calculate disagreement rates for true positives, true negatives, false positives, and false negatives.

In [ ]:
import pandas as pd
import numpy as np

print("NOTEBOOK 09 — STEP 8D: VIEW DISAGREEMENT AND CLASSIFICATION ERRORS")
print("-" * 70)

# --------------------------------------------------
# 1. Merge ground truth with view-consistency results
# --------------------------------------------------

analysis_df = consistency_df.merge(
    study_eval_df[
        ["study_id", "true_label", "predicted_class"]
    ],
    on="study_id",
    how="inner",
    suffixes=("_consistency", "_reference"),
    validate="one_to_one"
)

assert len(analysis_df) == 237

assert (
    analysis_df["predicted_class_consistency"]
    == analysis_df["predicted_class_reference"]
).all()

analysis_df = analysis_df.rename(
    columns={
        "predicted_class_reference": "predicted_class"
    }
)

analysis_df = analysis_df.drop(
    columns=["predicted_class_consistency"]
)

# --------------------------------------------------
# 2. Assign classification outcomes
# --------------------------------------------------

conditions = [
    (analysis_df["true_label"] == 1)
    & (analysis_df["predicted_class"] == 1),

    (analysis_df["true_label"] == 0)
    & (analysis_df["predicted_class"] == 0),

    (analysis_df["true_label"] == 0)
    & (analysis_df["predicted_class"] == 1),

    (analysis_df["true_label"] == 1)
    & (analysis_df["predicted_class"] == 0),
]

choices = ["TP", "TN", "FP", "FN"]

analysis_df["outcome"] = np.select(
    conditions,
    choices,
    default="UNKNOWN"
)

assert not (analysis_df["outcome"] == "UNKNOWN").any()

# --------------------------------------------------
# 3. Verify original confusion matrix
# --------------------------------------------------

outcome_counts = analysis_df["outcome"].value_counts()

expected_counts = {
    "TP": 88,
    "TN": 87,
    "FP": 53,
    "FN": 9
}

for outcome, expected in expected_counts.items():
    assert outcome_counts.get(outcome, 0) == expected

print("\nCLASSIFICATION OUTCOMES")
print(outcome_counts.reindex(["TP", "TN", "FP", "FN"]))

# --------------------------------------------------
# 4. Analyze disagreement by outcome
# --------------------------------------------------

summary = (
    analysis_df.groupby("outcome")
    .agg(
        studies=("study_id", "count"),
        disagreement_count=("threshold_disagreement", "sum"),
        disagreement_rate=("threshold_disagreement", "mean"),
        median_probability_range=("probability_range", "median"),
        median_probability_std=("probability_std", "median")
    )
    .reindex(["TP", "TN", "FP", "FN"])
)

summary["disagreement_percentage"] = (
    summary["disagreement_rate"] * 100
)

print("\nVIEW DISAGREEMENT BY CLASSIFICATION OUTCOME")
print(
    summary[
        [
            "studies",
            "disagreement_count",
            "disagreement_percentage",
            "median_probability_range",
            "median_probability_std"
        ]
    ].round(3).to_string()
)

# --------------------------------------------------
# 5. Inspect false-negative studies
# --------------------------------------------------

fn_df = (
    analysis_df[
        analysis_df["outcome"] == "FN"
    ]
    .sort_values(
        "probability_range",
        ascending=False
    )
)

print("\nFALSE-NEGATIVE STUDIES")
print(
    fn_df[
        [
            "study_id",
            "image_count",
            "mean_probability",
            "min_probability",
            "max_probability",
            "probability_range",
            "threshold_disagreement"
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# 6. Save analysis
# --------------------------------------------------

output_path = (
    AGENT_DIR
    / "evaluation"
    / "mura_official_view_consistency_error_analysis.csv"
)

analysis_df.to_csv(
    output_path,
    index=False
)

print("\nSaved to:", output_path)
print("\nStep 8D: PASSED")

# Step 8 — View Consistency: Summary and Conclusions

### Objective

Develop and validate a tool that measures disagreement between image-level predictions within the same MURA wrist radiographic study.

### Implementation

The `view_consistency()` function calculates:

- Minimum and maximum image-level abnormality probabilities.
- Mean probability.
- Probability range and standard deviation.
- Whether individual image predictions fall on opposite sides of the frozen classification threshold.

**Frozen threshold:** 0.0600893441587686

The tool does not modify the ResNet18 model, its predictions, or the study-level classification.

### Results — Official MURA Validation

The tool was evaluated on 659 radiographs representing 237 wrist studies.

| Metric | Result |
|---|---:|
| Total studies | 237 |
| Studies exhibiting threshold disagreement | 72 |
| Disagreement frequency | 30.38% |
| False negatives exhibiting disagreement | 5/9 (55.6%) |
| True negatives exhibiting disagreement | 25/87 (28.7%) |
| False positives exhibiting disagreement | 30/53 (56.6%) |
| True positives exhibiting disagreement | 12/88 (13.6%) |

### Key Findings

1. Image-level threshold disagreement was observed in approximately 30% of wrist studies.
2. Disagreement occurred more frequently among false-negative than true-negative studies.
3. Disagreement was also common among false-positive studies.
4. Four of nine false-negative studies showed no threshold disagreement.
5. Single-image studies cannot exhibit cross-image disagreement.

### Interpretation

View disagreement provides additional information about variation in model predictions within a study.

However, disagreement does not necessarily indicate incorrect classification, and agreement does not establish diagnostic correctness.

The tool measures **prediction consistency**, not independently verified anatomical projection consistency.

### Role in Agentic AI

The `view_consistency()` tool will provide evidence to the agent when deciding whether additional analysis may be useful.

Potential subsequent actions include:

- TTA-based prediction stability analysis.
- Grad-CAM visualization.
- Recommendation for human review.

These actions are not yet validated clinical decision rules.

### Conclusion

**Step 8 completed successfully.**

The view-consistency tool is implemented, unit-tested, and evaluated descriptively on the official MURA wrist validation dataset.

The next development stage is **Step 9 — TTA Uncertainty Analysis**.

# Step 9A — Test-Time Augmentation (TTA): Implementation

### Objective

Develop a tool to evaluate the stability of ResNet18 abnormality predictions under small, controlled image transformations.

### Method

Apply five deterministic image transformations:

1. Original image.
2. Brightness increased by 5%.
3. Brightness decreased by 5%.
4. Contrast increased by 5%.
5. Contrast decreased by 5%.

Each transformed image will be evaluated using the frozen ResNet18 classifier.

### Output

The tool will return:

- Original image probability.
- Individual augmented-image probabilities.
- Mean probability across augmentations.
- Standard deviation.
- Probability range.
- Whether augmented predictions cross the frozen classification threshold.

### Scientific Considerations

The TTA analysis is supplementary.

It will not change:

- ResNet18 model weights.
- The original study-level probability.
- The frozen classification threshold.
- The final study-level classification.

TTA variability measures sensitivity to the selected transformations. It does not establish calibrated predictive uncertainty.

The transformation magnitudes are initial engineering choices and have not been clinically validated.

### Agentic AI Integration

The agent may request TTA when additional information about prediction stability is needed.

The results will support an evidence-based report without overriding the frozen classifier.

In [ ]:
import io
import numpy as np
import torch

from PIL import Image, ImageEnhance


def tta_uncertainty(image_bytes, image_id="UNKNOWN"):
    """
    Evaluate ResNet18 prediction stability under
    small deterministic brightness/contrast changes.

    Does not modify the frozen classifier or threshold.
    """

    # ----------------------------------------
    # 1. Load image
    # ----------------------------------------

    image = Image.open(
        io.BytesIO(image_bytes)
    ).convert("RGB")

    # ----------------------------------------
    # 2. Define deterministic augmentations
    # ----------------------------------------

    augmented_images = {
        "original": image,
        "brightness_plus_5": ImageEnhance.Brightness(image).enhance(1.05),
        "brightness_minus_5": ImageEnhance.Brightness(image).enhance(0.95),
        "contrast_plus_5": ImageEnhance.Contrast(image).enhance(1.05),
        "contrast_minus_5": ImageEnhance.Contrast(image).enhance(0.95)
    }

    # ----------------------------------------
    # 3. Run frozen ResNet18
    # ----------------------------------------

    device = next(resnet18_model.parameters()).device

    resnet18_model.eval()

    predictions = {}

    with torch.inference_mode():

        for augmentation_name, augmented_image in augmented_images.items():

            tensor = inference_transform(
                augmented_image
            ).unsqueeze(0).to(device)

            logits = resnet18_model(tensor)

            probability = torch.sigmoid(
                logits
            ).reshape(-1).item()

            predictions[augmentation_name] = float(probability)

    # ----------------------------------------
    # 4. Calculate stability metrics
    # ----------------------------------------

    probabilities = np.array(
        list(predictions.values()),
        dtype=np.float64
    )

    original_probability = predictions["original"]

    threshold_classes = (
        probabilities >= SELECTED_THRESHOLD
    )

    threshold_disagreement = (
        len(np.unique(threshold_classes)) > 1
    )

    # ----------------------------------------
    # 5. Return structured results
    # ----------------------------------------

    return {
        "image_id": image_id,
        "status": "SUCCESS",
        "original_probability": original_probability,
        "tta_probabilities": predictions,
        "tta_mean_probability": float(np.mean(probabilities)),
        "tta_std": float(np.std(probabilities)),
        "tta_range": float(np.ptp(probabilities)),
        "threshold_disagreement": bool(threshold_disagreement),
        "threshold": float(SELECTED_THRESHOLD),
        "augmentation_count": len(predictions)
    }


print("Step 9A: TTA uncertainty tool defined successfully.")

## Step 9B — TTA Functional Testing

### Objective

Verify that the TTA uncertainty tool produces reproducible predictions using the frozen ResNet18 model.

### Method

Select one wrist radiograph from the official MURA validation dataset.

Apply five deterministic transformations:

- Original image
- Brightness +5%
- Brightness −5%
- Contrast +5%
- Contrast −5%

Compare the original-image probability with the previously saved ResNet18 prediction.

### Acceptance Criteria

1. Five augmentation probabilities are returned.
2. Every probability is finite and between 0 and 1.
3. The original-image prediction matches the frozen reference prediction within numerical tolerance.
4. TTA summary statistics are calculated correctly.
5. The frozen classification threshold remains unchanged.

### Scientific Considerations

TTA variability reflects sensitivity to the selected image transformations.

It is not a calibrated estimate of diagnostic uncertainty.

The original ResNet18 classification remains unchanged.

In [ ]:
import zipfile
import numpy as np

print("NOTEBOOK 09 — STEP 9B: TTA FUNCTIONAL TEST")
print("-" * 60)

# ---------------------------------------
# 1. Select one official wrist image
# ---------------------------------------

test_row = verification_df.iloc[0]

tta_test_image_id = str(test_row["image_id"])
tta_test_image_path = str(test_row["image_path"])
reference_probability = float(test_row["probability"])

print("\nImage ID:", tta_test_image_id)
print("Reference probability:", reference_probability)

# ---------------------------------------
# 2. Load original image bytes
# ---------------------------------------

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as zf:

    archive_names = zf.namelist()

    matches = [
        name for name in archive_names
        if name.endswith(tta_test_image_path.lstrip("/"))
    ]

    assert len(matches) == 1, (
        f"Expected one matching image, found {len(matches)}"
    )

    tta_test_image_bytes = zf.read(matches[0])

# ---------------------------------------
# 3. Run TTA
# ---------------------------------------

tta_result = tta_uncertainty(
    image_bytes=tta_test_image_bytes,
    image_id=tta_test_image_id
)

assert tta_result["status"] == "SUCCESS"
assert tta_result["augmentation_count"] == 5

# ---------------------------------------
# 4. Validate probabilities
# ---------------------------------------

probabilities = np.array(
    list(tta_result["tta_probabilities"].values()),
    dtype=np.float64
)

assert np.all(np.isfinite(probabilities))
assert np.all((probabilities >= 0) & (probabilities <= 1))

# ---------------------------------------
# 5. Verify frozen-model reproducibility
# ---------------------------------------

original_probability = tta_result["original_probability"]

difference = abs(
    original_probability - reference_probability
)

assert difference < 1e-5, (
    f"Original prediction mismatch: {difference}"
)

assert np.isclose(
    tta_result["tta_mean_probability"],
    np.mean(probabilities)
)

assert np.isclose(
    tta_result["tta_std"],
    np.std(probabilities)
)

assert np.isclose(
    tta_result["tta_range"],
    np.ptp(probabilities)
)

assert np.isclose(
    tta_result["threshold"],
    SELECTED_THRESHOLD
)

# ---------------------------------------
# 6. Display results
# ---------------------------------------

print("\nTTA PREDICTIONS")

for name, probability in tta_result["tta_probabilities"].items():
    print(f"{name:25s}: {probability:.6f}")

print("\nSTABILITY METRICS")
print(f"TTA mean probability: {tta_result['tta_mean_probability']:.6f}")
print(f"TTA standard deviation: {tta_result['tta_std']:.6f}")
print(f"TTA probability range: {tta_result['tta_range']:.6f}")

print(
    "Threshold disagreement:",
    tta_result["threshold_disagreement"]
)

print("\nREPRODUCIBILITY")
print(f"Reference probability: {reference_probability:.8f}")
print(f"Original probability:  {original_probability:.8f}")
print(f"Absolute difference:   {difference:.10f}")

print("\nStep 9B: PASSED")

## Step 9C — Structured TTA Decision Support

### Objective

Convert the numerical TTA results into structured information that can be used by the Agentic AI system.

### Method

The tool will examine:

1. Whether the TTA execution succeeded.
2. Whether augmented predictions cross the frozen classification threshold.
3. The magnitude of probability variation.
4. Whether the original ResNet18 prediction remains unchanged.

### Decision-Support Categories

**THRESHOLD_CROSSING**

At least one augmented prediction falls on the opposite side of the frozen threshold from another augmented prediction.

**NO_THRESHOLD_CROSSING**

All augmented predictions remain on the same side of the frozen threshold.

**UNAVAILABLE**

TTA results are missing or invalid.

### Scientific Considerations

These categories describe prediction behavior under selected transformations.

They do not establish diagnostic correctness or calibrated uncertainty.

The Agent may use this information when selecting additional analytical tools.

TTA must never override the original ResNet18 classification.

### Expected Output

A structured dictionary containing:

- TTA status.
- Stability category.
- Original probability.
- TTA probability range.
- TTA standard deviation.
- Frozen classification threshold.
- A descriptive explanation suitable for the final report.

In [ ]:
import numpy as np

def interpret_tta_result(tta_result):

    if not isinstance(tta_result, dict):
        return {
            "status": "UNAVAILABLE",
            "stability_category": "UNAVAILABLE",
            "reason": "TTA result is not a dictionary."
        }

    if tta_result.get("status") != "SUCCESS":
        return {
            "status": "UNAVAILABLE",
            "stability_category": "UNAVAILABLE",
            "reason": "TTA execution was unsuccessful."
        }

    try:
        predictions = tta_result["tta_probabilities"]

        required_augmentations = {
            "original",
            "brightness_plus_5",
            "brightness_minus_5",
            "contrast_plus_5",
            "contrast_minus_5"
        }

        if (
            not isinstance(predictions, dict)
            or set(predictions) != required_augmentations
        ):
            raise ValueError("Missing or unexpected augmentations.")

        probabilities = np.array(
            list(predictions.values()),
            dtype=float
        )

        if (
            not np.all(np.isfinite(probabilities))
            or np.any((probabilities < 0) | (probabilities > 1))
        ):
            raise ValueError("Invalid probability values.")

        threshold = float(tta_result["threshold"])

        if (
            not np.isfinite(threshold)
            or not np.isclose(
                threshold,
                SELECTED_THRESHOLD,
                rtol=0,
                atol=1e-12
            )
        ):
            raise ValueError("Frozen threshold mismatch.")

        original_probability = float(
            tta_result["original_probability"]
        )

        if not np.isclose(
            original_probability,
            predictions["original"],
            rtol=0,
            atol=1e-8
        ):
            raise ValueError("Original probability mismatch.")

        probability_range = float(np.ptp(probabilities))
        probability_std = float(np.std(probabilities))

        threshold_crossing = bool(
            np.any(probabilities >= threshold)
            and np.any(probabilities < threshold)
        )

        category = (
            "THRESHOLD_CROSSING"
            if threshold_crossing
            else "NO_THRESHOLD_CROSSING"
        )

        explanation = (
            "Augmented predictions crossed the frozen "
            "classification threshold."
            if threshold_crossing
            else
            "All augmented predictions remained on the "
            "same side of the frozen classification threshold."
        )

        return {
            "status": "SUCCESS",
            "stability_category": category,
            "original_probability": original_probability,
            "tta_probability_range": probability_range,
            "tta_probability_std": probability_std,
            "threshold": threshold,
            "threshold_crossing": threshold_crossing,
            "explanation": explanation,
            "classification_modified": False
        }

    except (KeyError, TypeError, ValueError, OverflowError) as error:
        return {
            "status": "UNAVAILABLE",
            "stability_category": "UNAVAILABLE",
            "reason": str(error)
        }


# Test using the result from Step 9B
tta_interpretation = interpret_tta_result(tta_result)

print("NOTEBOOK 09 — STEP 9C: TTA DECISION SUPPORT")
print("-" * 60)

for key, value in tta_interpretation.items():
    print(f"{key}: {value}")

assert tta_interpretation["status"] == "SUCCESS"
assert tta_interpretation["classification_modified"] is False

print("\nStep 9C: PASSED")

## Step 9D — TTA Tool Integration Preparation

### Objective

Prepare the validated TTA functions for reuse within the Agentic AI workflow.

### Components

**TTA uncertainty tool**

`tta_uncertainty()`

Performs five deterministic image transformations and calculates prediction stability metrics.

**TTA interpretation tool**

`interpret_tta_result()`

Produces structured information describing whether augmented predictions cross the frozen classification threshold.

### Integration Requirements

The TTA tool must:

- Use the frozen ResNet18 model.
- Preserve the original classification.
- Preserve the frozen decision threshold.
- Return structured results.
- Remain separate from the primary classification workflow.

### Scientific Limitations

TTA measures sensitivity to the selected transformations, not calibrated diagnostic uncertainty.

The Agent may use TTA findings to inform further analysis, but must not interpret prediction stability as proof of diagnostic correctness.

### Next Development Stage

Implement Grad-CAM to visualize image regions contributing to the ResNet18 prediction.

In [ ]:
from pathlib import Path

print("NOTEBOOK 09 — STEP 9D: AGENT TOOL DIRECTORY")
print("-" * 60)

TOOLS_DIR = AGENT_DIR / "tools"

TOOLS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Agent tools directory:", TOOLS_DIR)
print("Directory exists:", TOOLS_DIR.exists())

assert TOOLS_DIR.exists()

print("\nStep 9D: PASSED")

## Step 9E — Save TTA as a Reusable Python Module

### Objective

Package the TTA uncertainty and interpretation functions into a reusable Python module.

### Module

`MURA_AGENTIC_AI/tools/tta_tool.py`

### Design

The module contains two functions:

1. `tta_uncertainty()` — evaluates the frozen ResNet18 model under five deterministic image transformations.
2. `interpret_tta_result()` — converts TTA measurements into structured decision-support information.

The module receives the frozen model, preprocessing transform, and classification threshold as explicit dependencies.

### Validation

The saved module will be imported and tested against the previously validated notebook implementation.

### Constraints

- No model retraining.
- No modification of the frozen classification threshold.
- No modification of the primary study-level prediction.
- No clinical uncertainty claims based solely on TTA variability.

In [ ]:
import inspect
import textwrap
import importlib.util
import sys

print("NOTEBOOK 09 — STEP 9E: SAVE TTA MODULE")
print("-" * 60)

# Read the existing, tested function definitions
tta_source = textwrap.dedent(
    inspect.getsource(tta_uncertainty)
)

interpret_source = textwrap.dedent(
    inspect.getsource(interpret_tta_result)
)

# Replace notebook-global dependencies with explicit arguments
tta_source = tta_source.replace(
    'def tta_uncertainty(image_bytes, image_id="UNKNOWN"):',
    '''def tta_uncertainty(
    image_bytes,
    image_id="UNKNOWN",
    *,
    resnet18_model,
    inference_transform,
    selected_threshold
):'''
)

tta_source = tta_source.replace(
    "SELECTED_THRESHOLD",
    "selected_threshold"
)

interpret_source = interpret_source.replace(
    "def interpret_tta_result(tta_result):",
    "def interpret_tta_result(tta_result, *, selected_threshold):"
)

interpret_source = interpret_source.replace(
    "SELECTED_THRESHOLD",
    "selected_threshold"
)

module_header = '''
import io
import numpy as np
import torch

from PIL import Image, ImageEnhance

'''

module_path = TOOLS_DIR / "tta_tool.py"

module_path.write_text(
    module_header + tta_source + "\n\n" + interpret_source,
    encoding="utf-8"
)

print("Saved module:", module_path)

# Verify that the saved file is valid Python
source_code = module_path.read_text(encoding="utf-8")
compile(source_code, str(module_path), "exec")

print("Python syntax: VALID")

print("\nStep 9E: MODULE SAVED")

## Step 9F — Verification of the Saved TTA Module

### Objective

Verify that the reusable TTA module produces the same results as the previously tested notebook implementation.

### Method

1. Import `tta_tool.py` from Google Drive.
2. Run TTA on the same wrist radiograph used in Step 9B.
3. Compare the resulting probabilities with the original notebook results.
4. Verify that the interpretation function returns the same stability category.

### Acceptance Criteria

- The module imports successfully.
- All five TTA probabilities match the original implementation within numerical tolerance.
- The stability category is unchanged.
- The frozen classification threshold is preserved.
- The original classification is not modified.

### Expected Outcome

Successful verification will complete the implementation and packaging of the TTA tool.

The next stage will be Grad-CAM visualization.

In [ ]:
import importlib.util
import numpy as np

print("NOTEBOOK 09 — STEP 9F: VERIFY SAVED TTA MODULE")
print("-" * 60)

# ---------------------------------------
# 1. Import the saved module
# ---------------------------------------

module_path = TOOLS_DIR / "tta_tool.py"

spec = importlib.util.spec_from_file_location(
    "mura_tta_tool",
    module_path
)

tta_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tta_module)

print("Module imported successfully.")

# ---------------------------------------
# 2. Run the saved TTA implementation
# ---------------------------------------

module_result = tta_module.tta_uncertainty(
    image_bytes=tta_test_image_bytes,
    image_id=tta_test_image_id,
    resnet18_model=resnet18_model,
    inference_transform=inference_transform,
    selected_threshold=SELECTED_THRESHOLD
)

assert module_result["status"] == "SUCCESS"

# ---------------------------------------
# 3. Compare with the notebook results
# ---------------------------------------

print("\nTTA PROBABILITY COMPARISON")

for name, reference in tta_result["tta_probabilities"].items():

    actual = module_result["tta_probabilities"][name]

    difference = abs(actual - reference)

    print(
        f"{name:25s} "
        f"Difference: {difference:.10f}"
    )

    assert difference < 1e-5

# ---------------------------------------
# 4. Verify the interpretation function
# ---------------------------------------

module_interpretation = tta_module.interpret_tta_result(
    module_result,
    selected_threshold=SELECTED_THRESHOLD
)

assert module_interpretation["status"] == "SUCCESS"

assert (
    module_interpretation["stability_category"]
    == tta_interpretation["stability_category"]
)

assert module_interpretation["classification_modified"] is False

assert np.isclose(
    module_interpretation["threshold"],
    SELECTED_THRESHOLD
)

print("\nStability category:",
      module_interpretation["stability_category"])

print("Classification modified:",
      module_interpretation["classification_modified"])

print("\nStep 9F: PASSED")

## Step 10A — Grad-CAM: Model Architecture Inspection

### Objective

Prepare the frozen ResNet18 model for Grad-CAM visualization.

### Background

Grad-CAM uses gradients flowing into a convolutional layer to estimate which image regions contribute to a selected model output.

For binary classification, we will explain the model's abnormality logit.

### Method

1. Inspect the frozen ResNet18 architecture.
2. Identify the final convolutional layer.
3. Verify that the selected layer supports gradient-based analysis.

### Constraints

- Preserve the frozen model weights.
- Preserve the original classification threshold.
- Do not retrain or fine-tune the model.
- Do not interpret Grad-CAM heatmaps as confirmed lesion locations.

### Expected Outcome

Identification of a suitable convolutional layer for Grad-CAM implementation.

In [ ]:
import torch
import torch.nn as nn

print("NOTEBOOK 09 — STEP 10A: GRAD-CAM PREPARATION")
print("-" * 60)

# Verify the model architecture
print("\nModel type:")
print(type(resnet18_model).__name__)

# Inspect the final ResNet block
print("\nFinal convolutional block:")
print(resnet18_model.layer4[-1])

# Select the Grad-CAM target layer
gradcam_target_layer = resnet18_model.layer4[-1].conv2

print("\nSelected Grad-CAM target layer:")
print(gradcam_target_layer)

# Verify that it is convolutional
assert isinstance(
    gradcam_target_layer,
    nn.Conv2d
)

# Verify frozen parameters
trainable_parameters = sum(
    p.numel()
    for p in resnet18_model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", trainable_parameters)

assert trainable_parameters == 0, (
    "Model parameters must remain frozen."
)

print("\nStep 10A: PASSED")

## Step 10B — Grad-CAM Implementation

### Objective

Implement a Grad-CAM tool that visualizes image regions contributing to the frozen ResNet18 abnormality prediction.

### Method

1. Load and preprocess a wrist radiograph.
2. Capture activations from the final convolutional layer.
3. Calculate gradients of the abnormality logit with respect to those activations.
4. Compute gradient-based channel weights.
5. Generate a weighted activation map.
6. Apply ReLU and normalize the heatmap to the range [0, 1].
7. Resize the heatmap to the model input resolution.

### Model Configuration

- Architecture: ResNet18
- Target layer: `layer4[-1].conv2`
- Model input: 320 × 320 pixels
- Target output: Abnormality logit
- Model weights: Frozen

### Scientific Limitations

Grad-CAM identifies image regions contributing to a model output.

It does not independently identify fractures or other abnormalities.

Heatmaps are explanatory visualizations, not validated lesion segmentations.

### Agentic AI Integration

The Agent may request Grad-CAM when visual supporting information is useful.

Grad-CAM must not modify the primary classification or replace expert radiological interpretation.

In [ ]:
import inspect
import textwrap

print("NOTEBOOK 09 — STEP 10B: GRAD-CAM CORRECTION")
print("-" * 60)

# Retrieve the existing function source
source = textwrap.dedent(
    inspect.getsource(gradcam_explainer)
)

old_line = "input_tensor = input_tensor.to(device)"

new_line = (
    "input_tensor = input_tensor.to(device)"
    ".requires_grad_(True)"
)

assert source.count(old_line) == 1, (
    "Expected input tensor assignment not found exactly once."
)

# Apply correction
corrected_source = source.replace(
    old_line,
    new_line
)

# Redefine the function
exec(corrected_source, globals())

print("Grad-CAM function updated.")

# Verify model remains frozen
trainable_parameters = sum(
    p.numel()
    for p in resnet18_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_parameters)

assert trainable_parameters == 0

print("\nStep 10B correction: PASSED")

## Step 10C — Grad-CAM Functional Test

### Objective

Verify that the Grad-CAM tool produces a valid heatmap without changing the frozen ResNet18 prediction.

### Method

1. Run Grad-CAM on the previously tested wrist radiograph.
2. Verify that the returned heatmap has the expected dimensions.
3. Verify that heatmap values are finite and normalized to [0, 1].
4. Compare the Grad-CAM probability with the frozen reference prediction.
5. Display the original radiograph, heatmap, and overlay.

### Acceptance Criteria

- Grad-CAM executes successfully.
- Heatmap dimensions match the model input dimensions.
- Heatmap contains no NaN or infinite values.
- Heatmap values fall within [0, 1].
- The predicted probability matches the frozen model within numerical tolerance.
- No classification parameters are modified.

### Scientific Interpretation

The heatmap indicates regions contributing positively to the model's abnormality logit.

It does not establish whether a fracture or other lesion is present.

Visual inspection is required to identify obvious technical artifacts or uninformative heatmaps.

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("NOTEBOOK 09 — STEP 10C: GRAD-CAM FUNCTIONAL TEST")
print("-" * 60)

# 1. Generate Grad-CAM using the image from Step 9B

gradcam_result = gradcam_explainer(
    image_bytes=tta_test_image_bytes,
    image_id=tta_result["image_id"]
)

assert gradcam_result["status"] == "SUCCESS"

# 2. Verify heatmap

heatmap = gradcam_result["heatmap"]

expected_shape = (320, 320)

assert heatmap.shape == expected_shape, (
    f"Unexpected heatmap shape: {heatmap.shape}"
)

assert np.isfinite(heatmap).all()
assert heatmap.min() >= 0
assert heatmap.max() <= 1

# 3. Compare model probabilities

reference_probability = tta_result["original_probability"]
gradcam_probability = gradcam_result["probability"]

difference = abs(
    gradcam_probability - reference_probability
)

print("\nImage ID:", gradcam_result["image_id"])
print("Reference probability:", reference_probability)
print("Grad-CAM probability:", gradcam_probability)
print("Absolute difference:", difference)

assert difference < 1e-5, (
    "Grad-CAM prediction differs from the frozen model."
)

assert gradcam_result["classification_modified"] is False

# 4. Prepare visualization

original_image = Image.open(
    io.BytesIO(tta_test_image_bytes)
).convert("RGB")

display_image = original_image.resize(
    (320, 320)
)

# 5. Display three panels

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5)
)

axes[0].imshow(display_image)
axes[0].set_title("Original Wrist Radiograph")
axes[0].axis("off")

axes[1].imshow(
    heatmap,
    cmap="jet",
    vmin=0,
    vmax=1
)
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis("off")

axes[2].imshow(display_image)
axes[2].imshow(
    heatmap,
    cmap="jet",
    alpha=0.40,
    vmin=0,
    vmax=1
)
axes[2].set_title("Grad-CAM Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("\nHeatmap shape:", heatmap.shape)
print("Heatmap minimum:", float(heatmap.min()))
print("Heatmap maximum:", float(heatmap.max()))

print("\nStep 10C: PASSED")

## Step 10D — Save Grad-CAM as a Reusable Module

### Objective

Save the validated Grad-CAM implementation as a reusable Python module.

### Module

`MURA_AGENTIC_AI/tools/gradcam_tool.py`

### Requirements

The module must:

- Accept the frozen ResNet18 model as an explicit dependency.
- Accept the preprocessing transform and target convolutional layer.
- Calculate gradients without updating model parameters.
- Return a normalized heatmap.
- Preserve the original model prediction.
- Remove temporary forward hooks after execution.

### Validation

The saved module will be imported and tested against the original notebook implementation.

### Scientific Constraints

Grad-CAM is a model-attribution technique, not a validated fracture-localization method.

The resulting heatmap must not be used independently to establish a clinical diagnosis.

In [ ]:
import textwrap

print("NOTEBOOK 09 — STEP 10D: SAVE GRAD-CAM MODULE")
print("-" * 60)

# Use the corrected source created in Step 10B
gradcam_source = textwrap.dedent(corrected_source)

# Replace notebook-specific dependencies
old_signature = '''    model=resnet18_model,
    transform=inference_transform,
    target_layer=gradcam_target_layer'''

new_signature = '''    model,
    transform,
    target_layer'''

assert old_signature in gradcam_source, (
    "Expected Grad-CAM function signature not found."
)

gradcam_source = gradcam_source.replace(
    old_signature,
    new_signature
)

# Confirm gradient tracking is enabled
assert ".requires_grad_(True)" in gradcam_source

# Module imports
module_header = '''
import io
import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image

'''

# Save module
module_path = TOOLS_DIR / "gradcam_tool.py"

module_path.write_text(
    module_header + gradcam_source,
    encoding="utf-8"
)

# Validate Python syntax
source_code = module_path.read_text(encoding="utf-8")

compile(
    source_code,
    str(module_path),
    "exec"
)

print("Saved module:", module_path)
print("Python syntax: VALID")

print("\nStep 10D: MODULE SAVED")

## Step 10E — Verification of the Saved Grad-CAM Module

### Objective

Verify that the saved Grad-CAM module reproduces the results of the validated notebook implementation.

### Method

1. Import `gradcam_tool.py` from Google Drive.
2. Run the saved Grad-CAM function on the same wrist radiograph used in Step 10C.
3. Compare the predicted probability with the original Grad-CAM result.
4. Compare the generated heatmaps.
5. Confirm that model parameters remain frozen.

### Acceptance Criteria

- The module imports successfully.
- The predicted probability matches the original implementation within numerical tolerance.
- The heatmaps have identical dimensions.
- The maximum absolute heatmap difference is below 1e-5.
- Model parameters remain frozen.
- The classification is not modified.

### Expected Outcome

Successful verification completes the Grad-CAM tool implementation.

The next development stage is integration of the existing tools into the Agentic AI workflow.

In [ ]:
import importlib.util
import numpy as np

print("NOTEBOOK 09 — STEP 10E: VERIFY GRAD-CAM MODULE")
print("-" * 60)

# 1. Import saved module
module_path = TOOLS_DIR / "gradcam_tool.py"

spec = importlib.util.spec_from_file_location(
    "mura_gradcam_tool",
    module_path
)

gradcam_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gradcam_module)

print("Module imported successfully.")

# 2. Run saved implementation
module_gradcam_result = gradcam_module.gradcam_explainer(
    image_bytes=tta_test_image_bytes,
    image_id=gradcam_result["image_id"],
    model=resnet18_model,
    transform=inference_transform,
    target_layer=gradcam_target_layer
)

assert module_gradcam_result["status"] == "SUCCESS"

# 3. Compare probabilities
probability_difference = abs(
    module_gradcam_result["probability"]
    - gradcam_result["probability"]
)

print("\nProbability difference:", probability_difference)

assert probability_difference < 1e-5

# 4. Compare heatmaps
original_heatmap = gradcam_result["heatmap"]
module_heatmap = module_gradcam_result["heatmap"]

assert original_heatmap.shape == module_heatmap.shape

assert np.isfinite(module_heatmap).all()

heatmap_difference = np.max(
    np.abs(original_heatmap - module_heatmap)
)

print("Maximum heatmap difference:", heatmap_difference)

assert heatmap_difference < 1e-5

# 5. Verify frozen model
trainable_parameters = sum(
    p.numel()
    for p in resnet18_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_parameters)

assert trainable_parameters == 0

assert module_gradcam_result["classification_modified"] is False

print("\nStep 10E: PASSED")

# Step 10 — Grad-CAM Implementation and Validation

**Status: COMPLETED — ALL TESTS PASSED**

## 1. Objective

Implement, validate, and package Grad-CAM (Gradient-weighted Class Activation Mapping) as a reusable explainability tool for the MURA Agentic AI system.

Grad-CAM visualizes image regions that contribute to the frozen ResNet18 model's abnormality prediction.

The tool provides visual decision support without modifying the model's classification.

## 2. Model Configuration

| Parameter | Configuration |
|---|---|
| Model | Fine-tuned ResNet18 |
| Dataset | MURA v1.1 — XR_WRIST |
| Model status | Frozen |
| Target convolutional layer | `layer4[-1].conv2` |
| Model input | 320 × 320 pixels |
| Explained output | Abnormality logit |
| Classification threshold | 0.0600893441587686 |
| Model retraining | None |

The final convolutional layer was selected as the Grad-CAM target.

The model remained in evaluation mode, and all parameters retained `requires_grad=False`.

## 3. Grad-CAM Implementation

The `gradcam_explainer()` function performs the following operations:

1. Loads and preprocesses the radiographic image.
2. Enables gradient tracking for the input tensor.
3. Captures activations from the selected convolutional layer using a forward hook.
4. Calculates gradients of the abnormality logit with respect to the captured activations.
5. Computes channel weights

## Step 11A — Agent Workflow Preparation

### Objective

Prepare the integration of the validated MURA tools into a controlled Agentic AI workflow.

### Available Components

- Technical QC
- Anatomical Scope Validator
- Guarded ResNet18 Study Classification
- View Consistency
- TTA Uncertainty
- Grad-CAM Explanation

### Orchestration Principles

1. Technical QC and anatomical scope validation are mandatory.
2. Classification is permitted only after both guardrails pass.
3. Study classification uses the frozen ResNet18 model and frozen threshold.
4. View consistency is evaluated after classification.
5. TTA and Grad-CAM are optional decision-support tools.
6. Additional analyses must not modify the primary prediction.
7. The workflow must support human escalation and unsupported-input handling.

### Safety Requirements

The Agent must not bypass the guarded classification workflow.

Anatomical validation currently relies on MURA metadata and does not independently verify anatomy from image pixels.

The workflow must distinguish technical uncertainty, model prediction, and clinical interpretation.

### Next Step

Implement the Agent Orchestrator incrementally, beginning with verification that the required components are available.

In [ ]:
print("NOTEBOOK 09 — STEP 11A: AGENT TOOL INVENTORY")
print("-" * 60)

required_functions = {
    "Technical QC": "technical_qc",
    "Anatomical Scope Validator": "anatomical_scope_validator",
    "Guarded Classification": "guarded_classify_study",
    "View Consistency": "view_consistency",
}

for description, function_name in required_functions.items():

    available = (
        function_name in globals()
        and callable(globals()[function_name])
    )

    print(
        f"{description:30s}: "
        f"{'AVAILABLE' if available else 'MISSING'}"
    )

    assert available, (
        f"Required function missing: {function_name}"
    )

required_modules = {
    "TTA": TOOLS_DIR / "tta_tool.py",
    "Grad-CAM": TOOLS_DIR / "gradcam_tool.py",
}

for description, module_path in required_modules.items():

    available = module_path.is_file()

    print(
        f"{description:30s}: "
        f"{'AVAILABLE' if available else 'MISSING'}"
    )

    assert available, (
        f"Required module missing: {module_path}"
    )

print("\nStep 11A: PASSED")

## Step 11B — Agent Decision Framework

### Objective

Define a deterministic decision framework for the MURA Agentic AI Orchestrator.

### Available Actions

**routine_workflow**

Continue the standard workflow without an additional automated review flag.

**priority_human_review**

Flag a study for prioritized human review based on predefined workflow rules.

**human_escalation**

Escalate cases involving technical failures, missing results, or inconsistent tool outputs.

**unsupported_input**

Reject studies that are outside the validated anatomical scope.

### Decision Hierarchy

1. Unsupported anatomical scope → `unsupported_input`
2. Technical QC failure or missing required results → `human_escalation`
3. Valid abnormal prediction → `priority_human_review`
4. Valid normal prediction with image-level disagreement → `priority_human_review`
5. Valid normal prediction without disagreement → `routine_workflow`

### Scientific Constraints

These rules are initial engineering policies, not clinically validated triage criteria.

They must not be optimized using the official held-out validation dataset.

The Agent must preserve the frozen model prediction and threshold.

Optional TTA and Grad-CAM results may provide supporting information but must not override the primary classification.

### Validation Strategy

Use synthetic test cases to verify the decision hierarchy before integrating the decision function with the real tools.

In [ ]:
print("NOTEBOOK 09 — STEP 11B: AGENT DECISION FRAMEWORK")
print("-" * 60)


def agent_decision(
    qc_status,
    scope_status,
    classification_status,
    predicted_class,
    view_disagreement
):
    """
    Deterministic workflow decision policy.

    This function does not perform classification.
    It only determines the workflow action.
    """

    # 1. Validate required status values
    valid_qc = {"PASS", "WARNING", "FAIL"}
    valid_scope = {
        "SUPPORTED_WRIST",
        "UNSUPPORTED_ANATOMY",
        "UNCERTAIN_ANATOMY"
    }

    if (
        qc_status not in valid_qc
        or scope_status not in valid_scope
    ):
        return "human_escalation"

    # 2. Anatomical scope guardrail
    if scope_status != "SUPPORTED_WRIST":
        return "unsupported_input"

    # 3. Technical QC guardrail
    if qc_status != "PASS":
        return "human_escalation"

    # 4. Classification validity
    if classification_status != "SUCCESS":
        return "human_escalation"

    if predicted_class not in {0, 1}:
        return "human_escalation"

    if type(view_disagreement) is not bool:
        return "human_escalation"

    # 5. Abnormal prediction
    if predicted_class == 1:
        return "priority_human_review"

    # 6. Normal prediction with disagreement
    if view_disagreement:
        return "priority_human_review"

    # 7. Normal prediction without disagreement
    return "routine_workflow"


# ---------------------------------------
# Synthetic unit tests
# ---------------------------------------

test_cases = [
    (
        "Normal, consistent",
        "PASS", "SUPPORTED_WRIST", "SUCCESS",
        0, False, "routine_workflow"
    ),
    (
        "Abnormal prediction",
        "PASS", "SUPPORTED_WRIST", "SUCCESS",
        1, False, "priority_human_review"
    ),
    (
        "Normal, disagreement",
        "PASS", "SUPPORTED_WRIST", "SUCCESS",
        0, True, "priority_human_review"
    ),
    (
        "Unsupported anatomy",
        "PASS", "UNSUPPORTED_ANATOMY", "BLOCKED",
        None, None, "unsupported_input"
    ),
    (
        "Uncertain anatomy",
        "PASS", "UNCERTAIN_ANATOMY", "BLOCKED",
        None, None, "unsupported_input"
    ),
    (
        "QC failure",
        "FAIL", "SUPPORTED_WRIST", "BLOCKED",
        None, None, "human_escalation"
    ),
    (
        "Missing classification",
        "PASS", "SUPPORTED_WRIST", "ERROR",
        None, None, "human_escalation"
    ),
    (
        "QC warning",
        "WARNING", "SUPPORTED_WRIST", "BLOCKED",
        None, None, "human_escalation"
    )
]

for (
    name,
    qc,
    scope,
    classification,
    prediction,
    disagreement,
    expected
) in test_cases:

    result = agent_decision(
        qc_status=qc,
        scope_status=scope,
        classification_status=classification,
        predicted_class=prediction,
        view_disagreement=disagreement
    )

    print(f"{name:25s}: {result}")

    assert result == expected, (
        f"Failed test: {name}"
    )

print("\nStep 11B: PASSED")

## Step 11C — Guarded Classification Interface Inspection

### Objective

Inspect the existing guarded classification function before integrating it into the Agent Orchestrator.

### Rationale

The Orchestrator must use the validated classification interface and preserve the existing safety checks.

The integration must not:

- Bypass technical QC.
- Bypass anatomical scope validation.
- Modify the frozen ResNet18 model.
- Modify the frozen classification threshold.
- Introduce a second, inconsistent classification pathway.

### Method

Inspect the function signature and available documentation.

Identify the required inputs and the expected output structure.

### Expected Outcome

A verified interface specification that can be used to implement the Agent Orchestrator safely.

In [ ]:
import inspect

print("NOTEBOOK 09 — STEP 11C: GUARDED CLASSIFICATION INTERFACE")
print("-" * 60)

print("\nFunction:")
print(guarded_classify_study.__name__)

print("\nSignature:")
print(inspect.signature(guarded_classify_study))

print("\nDocumentation:")
print(
    inspect.getdoc(guarded_classify_study)
    or "No documentation available."
)

print("\nStep 11C: INTERFACE INSPECTION COMPLETE")

## Step 11D — Guarded Classification Output Inspection

### Objective

Identify the exact output structure of the validated `guarded_classify_study()` function.

### Method

1. Prepare a previously tested wrist radiograph.
2. Call the guarded classification function.
3. Inspect the returned dictionary and its nested fields.
4. Identify the outputs required by the Agent Orchestrator.

### Safety Requirements

- Use the existing guarded classification function.
- Preserve technical QC and anatomical scope validation.
- Preserve the frozen ResNet18 model and classification threshold.
- Do not modify the prediction or decision rules.

### Expected Outcome

A verified output schema for connecting guarded classification, view consistency, and Agent decision-making.

In [ ]:
print("NOTEBOOK 09 — STEP 11D: GUARDED CLASSIFICATION OUTPUT")
print("-" * 60)

from pprint import pprint

# Use the previously tested wrist image.
# The path is retrieved from the verified lookup table.

integration_image_id = tta_result["image_id"]

integration_row = verification_df.loc[
    verification_df["image_id"] == integration_image_id
].iloc[0]

integration_image_path = str(integration_row["image_path"])

# Extract the study identifier from the MURA path.
# Example: valid/XR_WRIST/patient11185/study1_positive/image1.png

path_parts = integration_image_path.replace("\\", "/").split("/")

study_index = next(
    i for i, part in enumerate(path_parts)
    if part.startswith("study")
)

integration_study_id = "/".join(
    path_parts[:study_index + 1]
)

integration_images = [
    {
        "image_id": integration_image_id,
        "image_bytes": tta_test_image_bytes
    }
]

integration_image_paths = [
    integration_image_path
]

print("\nStudy ID:", integration_study_id)
print("Image ID:", integration_image_id)
print("Image path:", integration_image_path)

# Run the existing guarded classifier

guarded_result = guarded_classify_study(
    study_id=integration_study_id,
    images=integration_images,
    image_paths=integration_image_paths
)

print("\nGUARDED CLASSIFICATION RESULT")
pprint(guarded_result)

print("\nRETURN TYPE:", type(guarded_result).__name__)

if isinstance(guarded_result, dict):
    print("TOP-LEVEL KEYS:", list(guarded_result.keys()))

print("\nStep 11D: OUTPUT INSPECTION COMPLETE")

## Step 11E — Initial Agent Orchestrator Implementation

### Objective

Implement the first integrated MURA Agentic AI workflow using the previously validated tools.

### Workflow

1. Receive a wrist radiographic study.
2. Execute guarded classification.
3. Extract technical QC and anatomical validation results.
4. If classification succeeds, evaluate image-level prediction consistency.
5. Apply the predefined Agent decision policy.
6. Return a structured workflow result.

### Mandatory Safety Controls

The Orchestrator must use `guarded_classify_study()` as its only classification pathway.

Technical QC and anatomical validation must not be bypassed.

If a required check fails, the Orchestrator must not produce a new classification.

### Decision Policy

The Agent uses the deterministic decision framework defined in Step 11B.

The policy distinguishes:

- `routine_workflow`
- `priority_human_review`
- `human_escalation`
- `unsupported_input`

These actions describe workflow handling, not clinical diagnoses.

### Scope and Limitations

The initial Orchestrator does not autonomously select optional tools.

TTA and Grad-CAM will be integrated after the mandatory workflow has been tested.

The frozen ResNet18 model and classification threshold remain unchanged.

### Expected Outcome

A functioning Orchestrator that connects the validated tools and produces a structured result suitable for subsequent Agentic AI development.

In [ ]:
print("NOTEBOOK 09 — STEP 11E: AGENT ORCHESTRATOR")
print("-" * 60)


def mura_agent_orchestrator(study_id, images, image_paths):
    """
    Initial deterministic MURA Agent Orchestrator.

    Mandatory workflow:
        Guarded classification
        -> View consistency
        -> Agent decision

    The guarded classifier internally performs:
        Technical QC
        -> Anatomical scope validation
        -> Frozen ResNet18 classification
    """

    # ---------------------------------------
    # 1. Mandatory guarded classification
    # ---------------------------------------

    guarded = guarded_classify_study(
        study_id=study_id,
        images=images,
        image_paths=image_paths
    )

    # ---------------------------------------
    # 2. Extract guardrail results
    # ---------------------------------------

    qc_results = guarded.get("qc_results", [])

    anatomy_result = guarded.get("anatomy_result") or {}

    scope_status = anatomy_result.get(
        "status", "UNKNOWN"
    )

    qc_statuses = [
        result.get("status", "UNKNOWN")
        for result in qc_results
    ]

    if not qc_statuses:
        qc_status = "UNKNOWN"
    elif "FAIL" in qc_statuses:
        qc_status = "FAIL"
    elif "WARNING" in qc_statuses:
        qc_status = "WARNING"
    elif all(status == "PASS" for status in qc_statuses):
        qc_status = "PASS"
    else:
        qc_status = "UNKNOWN"

    # ---------------------------------------
    # 3. Extract classification
    # ---------------------------------------

    classification = (
        guarded.get("classification_result") or {}
    )

    classification_status = classification.get(
        "status", "NOT_PERFORMED"
    )

    predicted_class = classification.get(
        "predicted_class"
    )

    study_probability = classification.get(
        "study_probability"
    )

    image_predictions = classification.get(
        "image_predictions", []
    )

    # ---------------------------------------
    # 4. View consistency
    # ---------------------------------------

    consistency_result = None
    view_disagreement = None

    if (
        guarded.get("status") == "SUCCESS"
        and guarded.get("classification_performed") is True
        and classification_status == "SUCCESS"
        and qc_status == "PASS"
        and scope_status == "SUPPORTED_WRIST"
    ):

        consistency_result = view_consistency(
            image_predictions,
            threshold=SELECTED_THRESHOLD
        )

        if consistency_result.get("status") == "SUCCESS":
            view_disagreement = bool(
                consistency_result["threshold_disagreement"]
            )

    # ---------------------------------------
    # 5. Agent decision
    # ---------------------------------------

    decision = agent_decision(
        qc_status=qc_status,
        scope_status=scope_status,
        classification_status=classification_status,
        predicted_class=predicted_class,
        view_disagreement=view_disagreement
    )

    # ---------------------------------------
    # 6. Structured output
    # ---------------------------------------

    return {
        "study_id": study_id,
        "workflow_status": guarded.get("status"),
        "qc_status": qc_status,
        "anatomy_status": scope_status,
        "classification_status": classification_status,
        "predicted_class": predicted_class,
        "study_probability": study_probability,
        "view_consistency": consistency_result,
        "agent_action": decision,
        "classification_modified": False
    }


print("Agent Orchestrator defined successfully.")
print("\nStep 11E: FUNCTION DEFINED")

## Step 11F — First End-to-End Agent Integration Test

### Objective

Verify that the MURA Agent Orchestrator correctly processes a complete wrist radiographic study.

### Test Procedure

1. Select a previously evaluated study from the official MURA wrist validation dataset.
2. Retrieve all images belonging to the study.
3. Execute the Agent Orchestrator.
4. Verify successful technical QC and anatomical scope validation.
5. Verify that the frozen ResNet18 classifier executes successfully.
6. Verify that view consistency is calculated.
7. Compare the study probability with the frozen reference prediction.
8. Verify that the Agent produces a valid workflow action.

### Acceptance Criteria

- All images belonging to the selected study are processed.
- Technical QC passes.
- Anatomical scope validation returns `SUPPORTED_WRIST`.
- Classification completes successfully.
- Study probability matches the frozen reference within numerical tolerance.
- View consistency is calculated successfully.
- The Agent returns a valid workflow action.
- The primary classification remains unchanged.

### Scientific Constraints

The official validation dataset is used only to verify software integration and numerical reproducibility.

No model retraining, threshold adjustment, or workflow-policy optimization will be performed.

The Agent's workflow action is not a clinical diagnosis or a validated urgency assessment.

In [ ]:
print("NOTEBOOK 09 — STEP 11F: STUDY ID CORRECTION")
print("-" * 60)

# Previously tested image
test_image_id = tta_result["image_id"]

print("\nImage ID:", test_image_id)

# Find the image in the verified prediction table
matching_rows = verification_df.loc[
    verification_df["image_id"] == test_image_id
]

assert len(matching_rows) == 1, (
    "Expected exactly one matching image."
)

# Retrieve the study ID used by the reference data
test_study_id = str(
    matching_rows.iloc[0]["study_id"]
)

print("\nStudy ID from verification_df:")
print(repr(test_study_id))

print("\nPreviously constructed study ID:")
print(repr(integration_study_id))

# Retrieve every image belonging to this study
study_rows = verification_df.loc[
    verification_df["study_id"] == test_study_id
].copy()

study_rows = study_rows.sort_values("image_id")

print("\nNumber of images in study:", len(study_rows))

print("\nStudy image IDs:")
print(study_rows["image_id"].tolist())

assert len(study_rows) > 0

print("\nStep 11F: STUDY IDENTIFICATION PASSED")

## Step 11F — Complete Study Integration Test

### Objective

Verify the first end-to-end execution of the MURA Agent Orchestrator using a complete four-image wrist study.

### Study Identification

- Reference study ID: `OFFICIAL_STUDY_00001`
- MURA study path: `valid/XR_WRIST/patient11185/study1_positive`
- Number of images: 4

The reference study ID is used consistently throughout the integration test.

### Workflow

1. Retrieve all four study images from the MURA archive.
2. Execute the Agent Orchestrator.
3. Verify technical QC and anatomical scope validation.
4. Verify successful frozen ResNet18 classification.
5. Verify image-level prediction consistency.
6. Compare the resulting study probability with the frozen reference.
7. Verify that the Agent returns a valid workflow action.

### Acceptance Criteria

- All four images are processed.
- Mandatory guardrails pass.
- The frozen study probability is reproduced.
- View consistency is calculated successfully.
- The Agent returns a valid workflow action.
- No model parameters, thresholds, or classifications are modified.

### Scientific Constraints

This test evaluates software integration and numerical reproducibility only.

It does not establish clinical validity or justify modifying the Agent's decision policy.

In [ ]:
import zipfile
import numpy as np

print("NOTEBOOK 09 — STEP 11F: COMPLETE INTEGRATION TEST")
print("-" * 60)

# 1. Use the corrected study identifier
test_study_id = "OFFICIAL_STUDY_00001"

study_rows = verification_df.loc[
    verification_df["study_id"] == test_study_id
].copy()

study_rows = study_rows.sort_values("image_id")

assert len(study_rows) == 4, (
    f"Expected 4 images, found {len(study_rows)}"
)

print("\nStudy ID:", test_study_id)
print("Images:", len(study_rows))

# 2. Load all study images
study_images = []
study_image_paths = []

with zipfile.ZipFile(MURA_ZIP_PATH, "r") as archive:

    archive_names = archive.namelist()

    for _, row in study_rows.iterrows():

        image_path = str(row["image_path"])

        matches = [
            name for name in archive_names
            if name.endswith(image_path)
        ]

        assert len(matches) == 1, (
            f"Archive match problem: {image_path}"
        )

        image_bytes = archive.read(matches[0])

        study_images.append({
            "image_id": row["image_id"],
            "image_bytes": image_bytes
        })

        study_image_paths.append(image_path)

print("Images loaded:", len(study_images))

# 3. Run the Agent Orchestrator
agent_result = mura_agent_orchestrator(
    study_id=test_study_id,
    images=study_images,
    image_paths=study_image_paths
)

# 4. Inspect results
print("\nAGENT WORKFLOW RESULTS")

for key in [
    "workflow_status",
    "qc_status",
    "anatomy_status",
    "classification_status",
    "predicted_class",
    "study_probability",
    "agent_action"
]:
    print(f"{key}: {agent_result.get(key)}")

# 5. Verify mandatory workflow
assert agent_result["workflow_status"] == "SUCCESS"
assert agent_result["qc_status"] == "PASS"
assert agent_result["anatomy_status"] == "SUPPORTED_WRIST"
assert agent_result["classification_status"] == "SUCCESS"

assert agent_result["view_consistency"] is not None
assert agent_result["view_consistency"]["status"] == "SUCCESS"

# 6. Compare against frozen reference
reference_row = reference_studies.loc[
    reference_studies["study_id"] == test_study_id
].iloc[0]

reference_probability = float(
    reference_row["mean_probability"]
)

agent_probability = float(
    agent_result["study_probability"]
)

difference = abs(
    reference_probability - agent_probability
)

print("\nREFERENCE COMPARISON")
print("Reference probability:", reference_probability)
print("Agent probability:", agent_probability)
print("Absolute difference:", difference)

assert difference < 1e-5

assert (
    agent_result["predicted_class"]
    == int(reference_row["predicted_class"])
)

# 7. Verify Agent decision
valid_actions = {
    "routine_workflow",
    "priority_human_review",
    "human_escalation",
    "unsupported_input"
}

assert agent_result["agent_action"] in valid_actions
assert agent_result["classification_modified"] is False

print("\nView disagreement:",
      agent_result["view_consistency"]["threshold_disagreement"])

print("Agent action:", agent_result["agent_action"])

print("\nStep 11F: PASSED")

## Step 11G — Agent Safety Validation

### Objective

Verify that the Agent Orchestrator respects the mandatory anatomical scope guardrail.

### Test Scenario

Submit a valid wrist radiograph with metadata indicating an unsupported anatomical region.

### Expected Behavior

1. Technical QC evaluates the image.
2. Anatomical scope validation identifies unsupported anatomy.
3. The guarded classifier blocks model inference.
4. The Agent returns `unsupported_input`.
5. No classification probability is generated.

### Acceptance Criteria

- Anatomical status: `UNSUPPORTED_ANATOMY`
- Classification not performed
- Agent action: `unsupported_input`
- No study probability
- No modification of model parameters

### Scientific Limitation

The anatomical guardrail validates supplied MURA metadata rather than independently recognizing anatomy from image pixels.

Therefore, this test verifies metadata-based workflow enforcement, not anatomical recognition.

### Next Stage

After safety validation, integrate optional TTA and Grad-CAM into the Agent workflow.

In [ ]:
print("NOTEBOOK 09 — STEP 11G: AGENT SAFETY TEST")
print("-" * 60)

# Use the same four images from Step 11F.
# Change only the declared anatomical region in their paths.

unsupported_paths = [
    path.replace("/XR_WRIST/", "/XR_SHOULDER/")
    for path in study_image_paths
]

assert all(
    "/XR_SHOULDER/" in path
    for path in unsupported_paths
)

# Execute the existing Agent Orchestrator

safety_result = mura_agent_orchestrator(
    study_id=test_study_id,
    images=study_images,
    image_paths=unsupported_paths
)

print("\nSAFETY TEST RESULTS")

for key in [
    "workflow_status",
    "qc_status",
    "anatomy_status",
    "classification_status",
    "predicted_class",
    "study_probability",
    "agent_action"
]:
    print(f"{key}: {safety_result.get(key)}")

# Verify that unsupported anatomy was blocked

assert (
    safety_result["anatomy_status"]
    == "UNSUPPORTED_ANATOMY"
)

assert (
    safety_result["agent_action"]
    == "unsupported_input"
)

assert safety_result["study_probability"] is None
assert safety_result["predicted_class"] is None

assert safety_result["classification_modified"] is False

print("\nStep 11G: PASSED")